# CUAD Clause Classification Experiments

This notebook contains the frozen primary experimental study for a 20-class CUAD clause-classification task. It evaluates five methods on the same document-title-group test split:

1. TF-IDF + balanced logistic regression
2. Zero-shot Mistral-7B-Instruct
3. PEFT (LoRA)
4. Retrieval-augmented generation (RAG)
5. PEFT + RAG hybrid

**Primary metric:** Macro F1.  
**Frozen split sizes:** 8,257 training, 833 validation, and 1,850 test clauses.  
**Execution environment:** Google Colab with a GPU for Mistral inference and PEFT training.

Run the notebook from top to bottom. Long-running inference cells save resumable checkpoints to Google Drive. The final section is authoritative; earlier exploratory and superseded analyses have been removed from this clean copy.


## 0. Environment, data, and shared model setup

The setup is split into dependency installation, dataset validation, and model initialization so failures are easier to diagnose. Select a GPU runtime before running these cells.


In [ ]:
# Install dependencies before importing experiment libraries.
from google.colab import drive

drive.mount("/content/drive")

%pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    trl \
    sentence-transformers \
    faiss-cpu \
    datasets


In [ ]:
# Imports
import os
import gc
import json
import zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# Paths and model configuration
DATA_PATH = '/content/drive/MyDrive/Thesis1_Project'
COLAB_WORKSPACE = '/content/cuad_data'
DRIVE_ZIP_PATH = os.path.join(DATA_PATH, 'CUAD.zip')
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'
FINAL_CLAUSE_TYPES = ['Parties', 'License Grant', 'Cap On Liability', 'Audit Rights', 'Anti-Assignment', 'Insurance', 'Document Name', 'Agreement Date', 'Governing Law', 'Expiration Date', 'Effective Date', 'Post-Termination Services', 'Revenue/Profit Sharing', 'Minimum Commitment', 'Exclusivity', 'Rofr/Rofo/Rofn', 'Ip Ownership Assignment', 'Non-Transferable License', 'Non-Compete', 'Termination For Convenience']
print(f'Project directory: {DATA_PATH}')
print(f'Base model: {MODEL_ID}')
print(f'Frozen classes: {len(FINAL_CLAUSE_TYPES)}')

# Check / extract cuad dataset if required
final_data_file = os.path.join(COLAB_WORKSPACE, 'CUAD_v1/CUAD_v1/master_clauses.csv')
if not os.path.exists(final_data_file):
    os.makedirs(COLAB_WORKSPACE, exist_ok=True)
    if not os.path.exists(DRIVE_ZIP_PATH):
        raise FileNotFoundError(f'CUAD.zip not found at:\n{DRIVE_ZIP_PATH}')
    print('\nExtracting CUAD.zip...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(COLAB_WORKSPACE)
    inner_zip_path = os.path.join(COLAB_WORKSPACE, 'CUAD_v1.zip')
    if os.path.exists(inner_zip_path):
        print('Extracting nested CUAD_v1.zip...')
        with zipfile.ZipFile(inner_zip_path, 'r') as z:
            z.extractall(os.path.join(COLAB_WORKSPACE, 'CUAD_v1'))

# Verify gpu
if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required.\nIn Google Colab go to:\nRuntime > Change runtime type > GPU')
print('\nGPU detected:')
get_ipython().system('nvidia-smi')

def print_gpu_memory():
    allocated = torch.cuda.memory_allocated() / 1024 ** 3
    reserved = torch.cuda.memory_reserved() / 1024 ** 3
    print(f'GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved')
gc.collect()
torch.cuda.empty_cache()

# Load dataset configuration
config_path = os.path.join(DATA_PATH, 'dataset_config.json')
if not os.path.exists(config_path):
    raise FileNotFoundError(f'dataset_config.json was not found at:\n{config_path}\n\nRun the final data-preparation notebook first.')
with open(config_path, 'r') as f:
    config = json.load(f)

# Load frozen class list from actual config structure
CLAUSE_TYPES = config.get('dataset_info', {}).get('final_clause_types', [])
if not CLAUSE_TYPES:
    raise ValueError('dataset_config.json does not contain:\ndataset_info -> final_clause_types')
if len(CLAUSE_TYPES) != 20:
    raise ValueError(f'Expected 20 frozen clause classes, but config contains {len(CLAUSE_TYPES)}.')
if CLAUSE_TYPES != FINAL_CLAUSE_TYPES:
    print('\nLabels stored in dataset_config.json:')
    for i, label in enumerate(CLAUSE_TYPES):
        print(i, label)
    print('\nExpected frozen labels:')
    for i, label in enumerate(FINAL_CLAUSE_TYPES):
        print(i, label)
    raise ValueError('The dataset configuration does not match the frozen 20-class thesis label set.')
print(f'✓ Frozen label set validated: {len(CLAUSE_TYPES)} classes')

# Validate label mappings
label_mapping = config.get('label_mapping', {})
config_label2id = label_mapping.get('label2id', {})
config_id2label = label_mapping.get('id2label', {})
if not config_label2id:
    raise ValueError("dataset_config.json is missing label_mapping['label2id'].")
if not config_id2label:
    raise ValueError("dataset_config.json is missing label_mapping['id2label'].")
expected_label2id = {label: i for i, label in enumerate(FINAL_CLAUSE_TYPES)}
expected_id2label_json = {str(i): label for i, label in enumerate(FINAL_CLAUSE_TYPES)}
if config_label2id != expected_label2id:
    print('\nConfig label2id:')
    print(config_label2id)
    print('\nExpected label2id:')
    print(expected_label2id)
    raise ValueError("label_mapping['label2id'] does not match the frozen label order.")
if config_id2label != expected_id2label_json:
    print('\nConfig id2label:')
    print(config_id2label)
    print('\nExpected id2label:')
    print(expected_id2label_json)
    raise ValueError("label_mapping['id2label'] does not match the frozen label order.")

# Rag configuration
rag_config = config.get('rag_config', {})
RAG_NORMALIZED = rag_config.get('normalized_embeddings', False)
print(f'✓ RAG normalized embeddings: {RAG_NORMALIZED}')

# Create experiment label variables
choices = FINAL_CLAUSE_TYPES.copy()
choices_str = ', '.join(choices)
label2id = {label: i for i, label in enumerate(choices)}
id2label = {i: label for i, label in enumerate(choices)}

# Load train / validation / test datasets
train_path = os.path.join(DATA_PATH, 'train.csv')
val_path = os.path.join(DATA_PATH, 'val.csv')
test_path = os.path.join(DATA_PATH, 'test.csv')
for path in [train_path, val_path, test_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Required dataset file not found:\n{path}')
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Validate required columns
required_columns = {'clause_text', 'clause_type', 'contract_title'}
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f'{name} dataset is missing columns: {sorted(missing_columns)}')

# Validate labels in all dataset splits
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    unknown = sorted(set(df['clause_type'].unique()) - set(choices))
    missing = sorted(set(choices) - set(df['clause_type'].unique()))
    if unknown or missing:
        raise ValueError(f'{name} labels invalid.\nUnknown labels: {unknown}\nMissing labels: {missing}')
    print(f"{name.title():12s}: {len(df):,} clauses | {df['contract_title'].nunique():,} document-title groups")

# Verify document-title-group separation
train_contracts = set(train_df['contract_title'])
val_contracts = set(val_df['contract_title'])
test_contracts = set(test_df['contract_title'])
train_val_overlap = train_contracts & val_contracts
train_test_overlap = train_contracts & test_contracts
val_test_overlap = val_contracts & test_contracts
if train_val_overlap or train_test_overlap or val_test_overlap:
    raise ValueError('Document-title-group leakage detected between train/validation/test splits.')


In [ ]:
# Import transformers after package installation
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Select safe compute dtype for tesla t4 / other gpus
COMPUTE_DTYPE = torch.float16

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)

# Load tokenizer
print(f'\nLoading tokenizer:\n{MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Load mistral 7b in 4-bit
print(f'\nLoading {MODEL_ID} in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True, dtype=COMPUTE_DTYPE)
model.eval()
model.config.use_cache = True

# Prompt construction helper
def build_prompt(instruction):
    messages = [{'role': 'user', 'content': instruction}]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f'<s>[INST] {instruction} [/INST]'

# Strict classification-output parser
def parse_prediction(raw_output, choices):
    """
    Accept only an exact class label.

    The parser deliberately does NOT search for class names
    inside arbitrary generated prose.
    """
    if raw_output is None:
        return 'Unknown'
    text = str(raw_output).strip()
    if not text:
        return 'Unknown'
    candidates = [text]
    lines = text.splitlines()
    if lines:
        candidates.append(lines[0].strip())
    for candidate in candidates:
        candidate = candidate.strip().strip('`*_').strip().strip('"').strip("'")
        if candidate.casefold().startswith('category:'):
            candidate = candidate.split(':', 1)[1].strip().strip('`*_').strip().strip('"').strip("'")
        for choice in choices:
            if candidate.casefold() == choice.casefold():
                return choice
    return 'Unknown'

# Evaluation helper
def evaluate_model(y_true, y_pred):
    return classification_report(y_true, y_pred, labels=choices, zero_division=0)

# Final setup summary
print(f'Frozen classes:      {len(choices)}')
print(f'Training clauses:    {len(train_df):,}')
print(f'Validation clauses:  {len(val_df):,}')
print(f'Test clauses:        {len(test_df):,}')
print(f'Model:               {MODEL_ID}')
print(f'RAG normalized:      {RAG_NORMALIZED}')
print(f'Compute dtype:       {COMPUTE_DTYPE}')
print_gpu_memory()


## 1. Zero-shot prompt engineering

Classify every test clause with the frozen base model and a constrained 20-label prompt. Format-normalized parsing accepts harmless punctuation while still treating non-label prose as `Unknown`.


In [ ]:
# Zero-shot inference and evaluation — part 1

import re

import shutil

from datetime import datetime, timezone

from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

RUN_TAG = 'v2_format_normalized'

partial_results_path = os.path.join(DATA_PATH, f'results_prompt_engineering_{RUN_TAG}_partial.csv')

versioned_final_path = os.path.join(DATA_PATH, f'results_prompt_engineering_{RUN_TAG}_final.csv')

canonical_final_path = os.path.join(DATA_PATH, 'results_prompt_engineering_final.csv')

previous_results_backup = os.path.join(DATA_PATH, 'results_prompt_engineering_pre_v2_backup.csv')

metadata_path = os.path.join(DATA_PATH, f'results_prompt_engineering_{RUN_TAG}_metadata.json')

if os.path.exists(canonical_final_path) and (not os.path.exists(previous_results_backup)):
    shutil.copy2(canonical_final_path, previous_results_backup)
    print(f'Previous results backed up to:\n{previous_results_backup}\n')

def parse_prediction(raw_output, choices):
    """
    Conservative output parser.

    Accepted:
    - Exact frozen class label
    - Capitalization differences
    - 'Category: <exact label>'
    - Surrounding quotation marks or markdown
    - Harmless terminal punctuation

    Rejected:
    - Synonyms
    - Unofficial category names
    - Multiple categories
    - Explanations containing a category
    - Arbitrary substring matches
    - Truncated labels
    """
    if raw_output is None or pd.isna(raw_output):
        return 'Unknown'
    text = str(raw_output).strip()
    if not text:
        return 'Unknown'
    candidates = [text]
    lines = text.splitlines()
    if lines:
        candidates.append(lines[0].strip())
    for candidate in candidates:
        candidate = candidate.strip()
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        category_match = re.fullmatch('(?i)category\\s*:\\s*(.+)', candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub('[.,;:!?]+$', '', candidate).strip()
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        for choice in choices:
            if candidate.casefold() == choice.casefold():
                return choice
    return 'Unknown'

experiment_test_df = test_df.reset_index(drop=True).copy()

experiment_test_df.insert(0, 'row_id', range(len(experiment_test_df)))

expected_columns = ['row_id', 'clause_text', 'clause_type', 'predicted_label', 'raw_output']

if os.path.exists(partial_results_path):
    completed_df = pd.read_csv(partial_results_path)
    missing_columns = set(expected_columns) - set(completed_df.columns)
    if missing_columns:
        raise ValueError(f'The existing V2 checkpoint has missing columns: {sorted(missing_columns)}')
    completed_df['row_id'] = pd.to_numeric(completed_df['row_id'], errors='raise').astype(int)
    completed_df = completed_df.drop_duplicates(subset='row_id', keep='last').sort_values('row_id').reset_index(drop=True)
    completed_ids = set(completed_df['row_id'])
    print(f'Existing V2 checkpoint found: {len(completed_df):,} completed predictions.')
else:
    completed_df = pd.DataFrame(columns=expected_columns)
    completed_ids = set()
    print('No V2 checkpoint found. Starting a clean run.')

remaining_df = experiment_test_df[~experiment_test_df['row_id'].isin(completed_ids)].copy().reset_index(drop=True)

print(f'Model:              {MODEL_ID}')

print(f'Frozen classes:     {len(choices)}')

print(f'Test clauses:       {len(experiment_test_df):,}')

print(f'Already completed:  {len(completed_df):,}')

print(f'Remaining clauses:  {len(remaining_df):,}')

print(f'Partial file:       {partial_results_path}')

def generate_prompt_predictions(batch_df):
    prediction_rows = []
    for row in batch_df.itertuples(index=False):
        instruction = f'Classify the legal clause below into exactly ONE of these categories:\n{choices_str}\n\nReturn ONLY the category name. Do not explain your answer and do not add punctuation.\n\nClause:\n{row.clause_text}'
        try:
            formatted_prompt = build_prompt(instruction)
            inputs = tokenizer(formatted_prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=12, do_sample=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
            generated_tokens = generated[0, inputs['input_ids'].shape[1]:]
            raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            predicted_label = parse_prediction(raw_output, choices)
            prediction_rows.append({'row_id': int(row.row_id), 'clause_text': row.clause_text, 'clause_type': row.clause_type, 'predicted_label': predicted_label, 'raw_output': raw_output})
        except RuntimeError as error:
            if 'out of memory' in str(error).lower():
                print(f'GPU out-of-memory error at row {row.row_id}. Recorded as Unknown.')
                torch.cuda.empty_cache()
                gc.collect()
                prediction_rows.append({'row_id': int(row.row_id), 'clause_text': row.clause_text, 'clause_type': row.clause_type, 'predicted_label': 'Unknown', 'raw_output': 'OOM'})
            else:
                raise
    return pd.DataFrame(prediction_rows, columns=expected_columns)

CHECKPOINT_INTERVAL = 20

for start in range(0, len(remaining_df), CHECKPOINT_INTERVAL):
    batch_df = remaining_df.iloc[start:start + CHECKPOINT_INTERVAL]
    batch_predictions = generate_prompt_predictions(batch_df)
    completed_df = pd.concat([completed_df, batch_predictions], ignore_index=True)
    completed_df['row_id'] = completed_df['row_id'].astype(int)
    completed_df = completed_df.drop_duplicates(subset='row_id', keep='last').sort_values('row_id').reset_index(drop=True)
    completed_df.to_csv(partial_results_path, index=False)
    print(f'Progress: {len(completed_df):,}/{len(experiment_test_df):,}')
    torch.cuda.empty_cache()
    gc.collect()

completed_df['row_id'] = completed_df['row_id'].astype(int)

completed_df = completed_df.drop_duplicates(subset='row_id', keep='last').sort_values('row_id').reset_index(drop=True)

if len(completed_df) != len(experiment_test_df):
    raise RuntimeError(f'Experiment is incomplete. Expected {len(experiment_test_df):,} predictions, but found {len(completed_df):,}.')

expected_row_ids = set(experiment_test_df['row_id'])

actual_row_ids = set(completed_df['row_id'])


In [ ]:
# Zero-shot inference and evaluation — part 2
if actual_row_ids != expected_row_ids:
    missing_ids = sorted(expected_row_ids - actual_row_ids)
    unexpected_ids = sorted(actual_row_ids - expected_row_ids)
    raise RuntimeError(f'Row-ID integrity check failed. Missing IDs: {missing_ids[:20]}; unexpected IDs: {unexpected_ids[:20]}')

if not completed_df['clause_type'].isin(choices).all():
    raise RuntimeError('Unexpected true labels found in the completed results.')

valid_prediction_values = set(choices) | {'Unknown'}

invalid_prediction_values = sorted(set(completed_df['predicted_label']) - valid_prediction_values)

if invalid_prediction_values:
    raise RuntimeError(f'Parser produced invalid prediction values: {invalid_prediction_values}')

y_true = completed_df['clause_type']

y_pred = completed_df['predicted_label']

accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)

macro_recall = recall_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)

macro_f1 = f1_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)

weighted_f1 = f1_score(y_true, y_pred, labels=choices, average='weighted', zero_division=0)

unknown_rate = completed_df['predicted_label'].eq('Unknown').mean()

unknown_count = completed_df['predicted_label'].eq('Unknown').sum()

report = classification_report(y_true, y_pred, labels=choices, zero_division=0)

completed_df.to_csv(versioned_final_path, index=False)

completed_df.to_csv(canonical_final_path, index=False)

completed_df.to_csv(partial_results_path, index=False)

run_metadata = {'experiment': 'Prompt Engineering — Zero-Shot', 'run_tag': RUN_TAG, 'model': MODEL_ID, 'test_samples': int(len(completed_df)), 'frozen_classes': int(len(choices)), 'generation': {'do_sample': False, 'max_new_tokens': 12, 'max_input_length': 4096}, 'parser': {'version': 'format_normalized_v2', 'accepts_exact_labels_only': True, 'allows_terminal_punctuation': True, 'allows_category_prefix': True, 'allows_synonyms': False, 'allows_substring_matching': False}, 'metrics': {'accuracy': float(accuracy), 'macro_precision': float(macro_precision), 'macro_recall': float(macro_recall), 'macro_f1': float(macro_f1), 'weighted_f1': float(weighted_f1), 'unknown_count': int(unknown_count), 'unknown_rate': float(unknown_rate)}, 'completed_at_utc': datetime.now(timezone.utc).isoformat()}

with open(metadata_path, 'w', encoding='utf-8') as metadata_file:
    json.dump(run_metadata, metadata_file, indent=2)

print(f'Total predictions:       {len(completed_df):,}')

print(f'Accuracy:                {accuracy:.4f}')

print(f'Macro precision:         {macro_precision:.4f}')

print(f'Macro recall:            {macro_recall:.4f}')

print(f'Macro F1:                {macro_f1:.4f}')

print(f'Weighted F1:             {weighted_f1:.4f}')

print(f'Unknown outputs:         {unknown_count:,}')

print(f'Unknown output rate:     {unknown_rate:.2%}')

print('\nClassification report:\n')

print(report)

print('\nMost frequent remaining Unknown outputs:')

remaining_unknown_outputs = completed_df.loc[completed_df['predicted_label'].eq('Unknown'), 'raw_output'].fillna('<EMPTY>').value_counts().head(30)

if len(remaining_unknown_outputs) == 0:
    print('None')
else:
    print(remaining_unknown_outputs.to_string())

print('\nFiles saved:')

print(f'  Versioned results: {versioned_final_path}')

print(f'  Canonical results: {canonical_final_path}')

print(f'  Resume checkpoint: {partial_results_path}')

print(f'  Run metadata:      {metadata_path}')


## 2. Parameter-efficient fine-tuning (PEFT)

Train a rank-16 LoRA adapter for one epoch on the frozen training split, then evaluate it on validation and test data. This is the frozen primary PEFT configuration; any later sensitivity analysis should be reported separately.


### 2A. Format the training and validation data


In [ ]:
# PEFT data formatting
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer
PEFT_CHECKPOINT_DIR = os.path.join(DATA_PATH, 'peft_v2_clean_checkpoints')
PEFT_ADAPTER_DIR = os.path.join(DATA_PATH, 'mistral_legal_adapter_v2_clean')
PEFT_COMPLETE_MARKER = os.path.join(PEFT_ADAPTER_DIR, 'training_complete.txt')
PEFT_VAL_PARTIAL = os.path.join(DATA_PATH, 'results_peft_v2_validation_partial.csv')
PEFT_TEST_PARTIAL = os.path.join(DATA_PATH, 'results_peft_v2_test_partial.csv')
PEFT_VAL_FINAL = os.path.join(DATA_PATH, 'results_peft_v2_validation_final.csv')
PEFT_TEST_VERSIONED = os.path.join(DATA_PATH, 'results_peft_v2_clean_final.csv')
PEFT_TEST_CANONICAL = os.path.join(DATA_PATH, 'results_peft_final.csv')
PEFT_REPORT_PATH = os.path.join(DATA_PATH, 'results_peft_v2_clean_report.txt')
PEFT_METRICS_PATH = os.path.join(DATA_PATH, 'results_peft_v2_clean_metrics.csv')
sft_train = load_dataset('json', data_files=os.path.join(DATA_PATH, 'peft_train.jsonl'), split='train')
sft_validation = load_dataset('json', data_files=os.path.join(DATA_PATH, 'peft_val.jsonl'), split='train')
required_columns = {'instruction', 'input', 'output'}
for split_name, dataset in [('training', sft_train), ('validation', sft_validation)]:
    missing = required_columns - set(dataset.column_names)
    if missing:
        raise ValueError(f'{split_name.title()} dataset is missing columns: {sorted(missing)}')

def format_sft_example(example):
    instruction = f"{example['instruction']}\n\nClause:\n{example['input']}\n\nReturn ONLY the category name. Do not explain your answer and do not add punctuation."
    return {'text': build_prompt(instruction) + ' ' + str(example['output']).strip() + tokenizer.eos_token}
sft_train = sft_train.map(format_sft_example, desc='Formatting training data')
sft_validation = sft_validation.map(format_sft_example, desc='Formatting validation data')
print(f'PEFT data ready: {len(sft_train):,} training and {len(sft_validation):,} validation examples.')


### 2B. Train and save the LoRA adapter


In [ ]:
# PEFT training
adapter_config_file = os.path.join(PEFT_ADAPTER_DIR, 'adapter_config.json')
training_complete = os.path.exists(PEFT_COMPLETE_MARKER) and os.path.exists(adapter_config_file)
if training_complete:
    print('Clean PEFT adapter already trained; training skipped.')
else:
    for variable_name in ['model', 'trainer', 'base_model']:
        if variable_name in globals():
            del globals()[variable_name]
    gc.collect()
    torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float16)
    if hasattr(model, 'peft_config'):
        raise RuntimeError('The fresh base model already contains a PEFT adapter.')
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], bias='none', task_type='CAUSAL_LM')
    training_args = SFTConfig(output_dir=PEFT_CHECKPOINT_DIR, dataset_text_field='text', max_length=2048, packing=False, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16, learning_rate=0.0002, num_train_epochs=1, eval_strategy='steps', eval_steps=100, save_strategy='steps', save_steps=100, save_total_limit=2, logging_steps=10, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, fp16=True, bf16=False, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, optim='paged_adamw_8bit', report_to=[], seed=42, data_seed=42)
    trainer = SFTTrainer(model=model, args=training_args, train_dataset=sft_train, eval_dataset=sft_validation, processing_class=tokenizer, peft_config=lora_config)
    if not hasattr(trainer.model, 'peft_config') or len(trainer.model.peft_config) != 1:
        raise RuntimeError('Expected exactly one LoRA adapter.')
    for parameter in trainer.model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.to(torch.float32)
    trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
    if trainable_dtypes != {torch.float32}:
        raise RuntimeError(f'Unexpected trainable parameter dtypes: {trainable_dtypes}')
    trainer.model.print_trainable_parameters()
    print('Trainable parameter dtype:', next((parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad)))
    last_checkpoint = get_last_checkpoint(PEFT_CHECKPOINT_DIR) if os.path.isdir(PEFT_CHECKPOINT_DIR) else None
    if last_checkpoint:
        print(f'Resuming from: {last_checkpoint}')
    else:
        print('Starting clean PEFT training.')
    trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.save_model(PEFT_ADAPTER_DIR)
    tokenizer.save_pretrained(PEFT_ADAPTER_DIR)
    os.makedirs(PEFT_ADAPTER_DIR, exist_ok=True)
    with open(PEFT_COMPLETE_MARKER, 'w', encoding='utf-8') as marker:
        marker.write('complete')
    print(f'Training complete. Adapter saved to {PEFT_ADAPTER_DIR}')


### 2C. Load the adapter for deterministic evaluation


In [ ]:
# PEFT evaluation setup
if not os.path.exists(PEFT_COMPLETE_MARKER):
    raise RuntimeError('PEFT training has not completed successfully.')
for variable_name in ['trainer', 'model', 'base_model']:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, PEFT_ADAPTER_DIR, is_trainable=False)
if not hasattr(model, 'peft_config') or len(model.peft_config) != 1:
    raise RuntimeError('Expected exactly one evaluation adapter.')
model.eval()
model.config.use_cache = True

def parse_prediction(raw_output, choices):
    """Accept only exact frozen labels with harmless formatting."""
    if raw_output is None or pd.isna(raw_output):
        return 'Unknown'
    text = str(raw_output).strip()
    if not text:
        return 'Unknown'
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        category_match = re.fullmatch('(?i)category\\s*:\\s*(.+)', candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub('[.,;:!?]+$', '', candidate).strip()
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        for choice in choices:
            if candidate.casefold() == choice.casefold():
                return choice
    return 'Unknown'

def evaluate_peft_split(source_df, split_name, partial_path, save_every=20):
    evaluation_df = source_df.reset_index(drop=True).copy()
    evaluation_df.insert(0, 'row_id', range(len(evaluation_df)))
    result_columns = ['row_id', 'clause_text', 'clause_type', 'predicted_label_peft', 'raw_output_peft']
    if os.path.exists(partial_path):
        completed = pd.read_csv(partial_path)
        if not set(result_columns).issubset(completed.columns):
            raise ValueError(f'The {split_name} checkpoint is incompatible.')
        completed['row_id'] = completed['row_id'].astype(int)
        completed = completed.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
    else:
        completed = pd.DataFrame(columns=result_columns)
    completed_ids = set(completed['row_id']) if len(completed) else set()
    remaining = evaluation_df[~evaluation_df['row_id'].isin(completed_ids)].reset_index(drop=True)
    print(f'{split_name.title()}: {len(completed):,} completed, {len(remaining):,} remaining.')
    for start in range(0, len(remaining), save_every):
        batch = remaining.iloc[start:start + save_every]
        batch_results = []
        for row in batch.itertuples(index=False):
            instruction = f'Classify the legal clause below into exactly ONE of these categories:\n{choices_str}\n\nReturn ONLY the category name. Do not explain your answer and do not add punctuation.\n\nClause:\n{row.clause_text}'
            try:
                inputs = tokenizer(build_prompt(instruction), return_tensors='pt', truncation=True, max_length=4096).to('cuda')
                with torch.inference_mode():
                    output = model.generate(**inputs, max_new_tokens=12, do_sample=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
                raw_output = tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
                prediction = parse_prediction(raw_output, choices)
            except RuntimeError as error:
                if 'out of memory' not in str(error).lower():
                    raise
                raw_output = 'OOM'
                prediction = 'Unknown'
                gc.collect()
                torch.cuda.empty_cache()
            batch_results.append({'row_id': int(row.row_id), 'clause_text': row.clause_text, 'clause_type': row.clause_type, 'predicted_label_peft': prediction, 'raw_output_peft': raw_output})
        completed = pd.concat([completed, pd.DataFrame(batch_results, columns=result_columns)], ignore_index=True)
        completed['row_id'] = completed['row_id'].astype(int)
        completed = completed.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
        completed.to_csv(partial_path, index=False)
        if len(completed) % 100 == 0 or len(completed) == len(evaluation_df):
            print(f'{split_name.title()}: {len(completed):,}/{len(evaluation_df):,}')
        gc.collect()
        torch.cuda.empty_cache()
    if len(completed) != len(evaluation_df):
        raise RuntimeError(f'{split_name.title()} evaluation is incomplete.')
    if set(completed['row_id']) != set(evaluation_df['row_id']):
        raise RuntimeError(f'{split_name.title()} row-ID check failed.')
    return completed
print('PEFT adapter loaded and evaluation functions ready.')


### 2D. Evaluate validation and test performance


In [ ]:
# PEFT validation and test evaluation
validation_predictions = evaluate_peft_split(val_df, 'validation', PEFT_VAL_PARTIAL)
test_predictions = evaluate_peft_split(test_df, 'test', PEFT_TEST_PARTIAL)

def add_predictions_to_source(source_df, predictions):
    source = source_df.reset_index(drop=True).copy()
    source.insert(0, 'row_id', range(len(source)))
    prediction_columns = predictions[['row_id', 'predicted_label_peft', 'raw_output_peft']]
    return source.merge(prediction_columns, on='row_id', how='left', validate='one_to_one')
validation_results = add_predictions_to_source(val_df, validation_predictions)
test_results = add_predictions_to_source(test_df, test_predictions)

def calculate_metrics(results):
    y_true = results['clause_type']
    y_pred = results['predicted_label_peft']
    return {'accuracy': accuracy_score(y_true, y_pred), 'macro_precision': precision_score(y_true, y_pred, labels=choices, average='macro', zero_division=0), 'macro_recall': recall_score(y_true, y_pred, labels=choices, average='macro', zero_division=0), 'macro_f1': f1_score(y_true, y_pred, labels=choices, average='macro', zero_division=0), 'weighted_f1': f1_score(y_true, y_pred, labels=choices, average='weighted', zero_division=0), 'unknown_count': int(y_pred.eq('Unknown').sum()), 'unknown_rate': y_pred.eq('Unknown').mean()}
validation_metrics = calculate_metrics(validation_results)
test_metrics = calculate_metrics(test_results)
validation_report = classification_report(validation_results['clause_type'], validation_results['predicted_label_peft'], labels=choices, zero_division=0)
test_report = classification_report(test_results['clause_type'], test_results['predicted_label_peft'], labels=choices, zero_division=0)
validation_results.to_csv(PEFT_VAL_FINAL, index=False)
test_results.to_csv(PEFT_TEST_VERSIONED, index=False)
test_results.to_csv(PEFT_TEST_CANONICAL, index=False)
metrics_table = pd.DataFrame([{'split': 'validation', **validation_metrics}, {'split': 'test', **test_metrics}])
metrics_table.to_csv(PEFT_METRICS_PATH, index=False)
with open(PEFT_REPORT_PATH, 'w', encoding='utf-8') as report_file:
    report_file.write('PEFT VALIDATION RESULTS\n' + '=' * 70 + '\n' + validation_report + '\n\nPEFT TEST RESULTS\n' + '=' * 70 + '\n' + test_report)
display(metrics_table[['split', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'unknown_count', 'unknown_rate']].round(4))
print('\nTest-set classification report:\n')
print(test_report)
print(f'\nAdapter: {PEFT_ADAPTER_DIR}')
print(f'Final results: {PEFT_TEST_CANONICAL}')


## 3. Retrieval-augmented generation (RAG)

Audit the training-only FAISS index, precompute top-3 retrievals, run Mistral with retrieved labeled demonstrations, and evaluate both performance and exact-text-match sensitivity.


### 3A. Audit the retrieval index


In [ ]:
# RAG index audit
import pickle
from collections import Counter
import faiss
from sentence_transformers import SentenceTransformer

# Configuration
RAG_K = 3
RAG_INDEX_PATH = os.path.join(DATA_PATH, 'rag_index.faiss')
RAG_METADATA_PATH = os.path.join(DATA_PATH, 'rag_metadata.pkl')

# Remove models left by earlier experiments
for variable_name in ['model', 'base_model', 'trainer', 'embedding_model']:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

# Load faiss index and metadata
if not os.path.exists(RAG_INDEX_PATH):
    raise FileNotFoundError(f'FAISS index not found:\n{RAG_INDEX_PATH}')
if not os.path.exists(RAG_METADATA_PATH):
    raise FileNotFoundError(f'RAG metadata not found:\n{RAG_METADATA_PATH}')
rag_index = faiss.read_index(RAG_INDEX_PATH)
with open(RAG_METADATA_PATH, 'rb') as metadata_file:
    rag_metadata = pickle.load(metadata_file)
rag_metadata_df = pd.DataFrame(rag_metadata).reset_index(drop=True)

# Basic structural checks
required_metadata_columns = {'clause_text', 'clause_type'}
missing_columns = required_metadata_columns - set(rag_metadata_df.columns)
if missing_columns:
    raise ValueError(f'RAG metadata is missing columns: {sorted(missing_columns)}')
if rag_index.ntotal != len(rag_metadata_df):
    raise ValueError(f'FAISS/metadata size mismatch: {rag_index.ntotal:,} vectors versus {len(rag_metadata_df):,} metadata records.')
if len(rag_metadata_df) != len(train_df):
    raise ValueError('RAG metadata does not contain exactly the training-set records.')
unknown_metadata_labels = sorted(set(rag_metadata_df['clause_type']) - set(choices))
if unknown_metadata_labels:
    raise ValueError(f'Unknown labels found in RAG metadata: {unknown_metadata_labels}')

# Verify that metadata is exactly the training set
def create_pair_counter(dataframe):
    return Counter(zip(dataframe['clause_text'].astype(str), dataframe['clause_type'].astype(str)))
metadata_pairs = create_pair_counter(rag_metadata_df)
training_pairs = create_pair_counter(train_df)
if metadata_pairs != training_pairs:
    extra_records = sum((metadata_pairs - training_pairs).values())
    missing_records = sum((training_pairs - metadata_pairs).values())
    raise ValueError(f'RAG metadata is not an exact multiset match for the training data.\nExtra records: {extra_records}\nMissing records: {missing_records}')

# Verify faiss configuration
index_metric = getattr(rag_index, 'metric_type', None)
if index_metric != faiss.METRIC_L2:
    raise ValueError('Expected an L2 FAISS index.')
if bool(RAG_NORMALIZED):
    raise ValueError('This index uses unnormalized embeddings, but RAG_NORMALIZED is set to True.')

# Load embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
embedding_dimension = embedding_model.get_embedding_dimension()
if rag_index.d != embedding_dimension:
    raise ValueError(f'Embedding dimension mismatch: index={rag_index.d}, model={embedding_dimension}')

# Check index-vector/metadata alignment
audit_positions = np.linspace(0, len(rag_metadata_df) - 1, num=min(5, len(rag_metadata_df)), dtype=int)
audit_texts = rag_metadata_df.iloc[audit_positions]['clause_text'].astype(str).tolist()
recomputed_vectors = embedding_model.encode(audit_texts, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype('float32')
stored_vectors = np.vstack([rag_index.reconstruct(int(position)) for position in audit_positions]).astype('float32')
alignment_distances = np.sum((recomputed_vectors - stored_vectors) ** 2, axis=1)
maximum_alignment_distance = float(alignment_distances.max())
if maximum_alignment_distance > 0.01:
    raise ValueError(f'The FAISS vectors may not align with the metadata or embedding model. Maximum audit distance: {maximum_alignment_distance:.6f}')

# Retrieval probe
probe_text = str(test_df.iloc[0]['clause_text'])
probe_vector = embedding_model.encode([probe_text], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype('float32')
probe_distances, probe_indices = rag_index.search(probe_vector, RAG_K)
probe_results = rag_metadata_df.iloc[probe_indices[0]][['clause_type', 'clause_text']].copy()
probe_results.insert(0, 'distance', probe_distances[0])

# Summary
print(f'FAISS vectors:          {rag_index.ntotal:,}')
print(f'Metadata records:       {len(rag_metadata_df):,}')
print(f'Training clauses:       {len(train_df):,}')
print(f'Non-training records:   0')
print(f'Index type:             {type(rag_index).__name__}')
print(f'Distance metric:        L2')
print(f'Embedding dimension:    {embedding_dimension}')
print(f'Normalized embeddings:  {RAG_NORMALIZED}')
print(f'Retrieval K:            {RAG_K}')
print(f'Maximum alignment distance: {maximum_alignment_distance:.8f}')
print('\nRetrieval probe:')
display(probe_results)


### 3B. Precompute test-set retrievals


In [ ]:
# Retrieval precomputation
RAG_RETRIEVAL_CACHE = os.path.join(DATA_PATH, 'rag_v2_clean_test_retrievals.csv')
RAG_RETRIEVAL_METRICS = os.path.join(DATA_PATH, 'rag_v2_clean_retrieval_metrics.json')

# Prepare stable test row ids
rag_test_df = test_df.reset_index(drop=True).copy()
rag_test_df.insert(0, 'row_id', range(len(rag_test_df)))

# Encode all test clauses
query_vectors = embedding_model.encode(rag_test_df['clause_text'].astype(str).tolist(), batch_size=64, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True).astype('float32')
if query_vectors.shape != (len(rag_test_df), rag_index.d):
    raise ValueError(f'Unexpected query-embedding shape: {query_vectors.shape}')

# Retrieve top-k training examples
retrieval_distances, retrieval_indices = rag_index.search(query_vectors, RAG_K)
retrieval_records = []
for row_position, row in rag_test_df.iterrows():
    neighbour_indices = [int(index) for index in retrieval_indices[row_position] if index >= 0]
    neighbour_distances = [float(distance) for distance in retrieval_distances[row_position][:len(neighbour_indices)]]
    neighbours = rag_metadata_df.iloc[neighbour_indices]
    neighbour_labels = neighbours['clause_type'].astype(str).tolist()
    neighbour_texts = neighbours['clause_text'].astype(str).tolist()
    correct_label_in_top3 = row['clause_type'] in neighbour_labels
    exact_text_match = any((str(row['clause_text']).strip() == retrieved_text.strip() for retrieved_text in neighbour_texts))
    retrieval_records.append({'row_id': int(row['row_id']), 'clause_type': row['clause_type'], 'top1_retrieved_label': neighbour_labels[0], 'correct_label_in_top3': bool(correct_label_in_top3), 'exact_text_match': bool(exact_text_match), 'neighbor_indices': json.dumps(neighbour_indices), 'neighbor_labels': json.dumps(neighbour_labels), 'neighbor_distances': json.dumps(neighbour_distances)})
retrieval_cache = pd.DataFrame(retrieval_records)
retrieval_cache.to_csv(RAG_RETRIEVAL_CACHE, index=False)

# Retrieval diagnostics
top1_label_accuracy = (retrieval_cache['top1_retrieved_label'] == retrieval_cache['clause_type']).mean()
correct_label_recall_at_3 = retrieval_cache['correct_label_in_top3'].mean()
exact_text_match_rate = retrieval_cache['exact_text_match'].mean()
retrieval_metrics = {'test_clauses': int(len(retrieval_cache)), 'k': int(RAG_K), 'top1_retrieved_label_accuracy': float(top1_label_accuracy), 'correct_label_recall_at_3': float(correct_label_recall_at_3), 'exact_text_match_count': int(retrieval_cache['exact_text_match'].sum()), 'exact_text_match_rate': float(exact_text_match_rate)}
with open(RAG_RETRIEVAL_METRICS, 'w', encoding='utf-8') as metrics_file:
    json.dump(retrieval_metrics, metrics_file, indent=2)
print(f'Test clauses:                 {len(retrieval_cache):,}')
print(f'Top-1 label accuracy:         {top1_label_accuracy:.4f}')
print(f'Correct-label recall@3:       {correct_label_recall_at_3:.4f}')
print(f"Exact text matches:           {retrieval_cache['exact_text_match'].sum():,}")
print(f'Exact text-match rate:        {exact_text_match_rate:.2%}')
print(f'Retrieval cache:              {RAG_RETRIEVAL_CACHE}')
print('\nMost frequently retrieved top-1 labels:')
display(retrieval_cache['top1_retrieved_label'].value_counts().rename_axis('label').reset_index(name='count'))


### 3C. Run RAG inference


In [ ]:
# RAG inference
RAG_PARTIAL_PATH = os.path.join(DATA_PATH, 'results_rag_v2_clean_partial.csv')

# Remove embedding model and free gpu memory
for variable_name in ['embedding_model', 'model', 'base_model', 'trainer']:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

# Load untouched base mistral model
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float16)
if hasattr(model, 'peft_config'):
    raise RuntimeError('Experiment 3 must use the untouched base model, but a PEFT adapter was detected.')
model.eval()
model.config.use_cache = True
tokenizer.padding_side = 'left'
tokenizer.truncation_side = 'left'

# Conservative output parser
def parse_prediction(raw_output, choices):
    """
    Accept an exact frozen class label with harmless
    formatting differences.
    """
    if raw_output is None or pd.isna(raw_output):
        return 'Unknown'
    text = str(raw_output).strip()
    if not text:
        return 'Unknown'
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        category_match = re.fullmatch('(?i)category\\s*:\\s*(.+)', candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub('[.,;:!?]+$', '', candidate).strip()
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        for choice in choices:
            if candidate.casefold() == choice.casefold():
                return choice
    return 'Unknown'

# Prompt-length control
def truncate_clause_for_prompt(text, maximum_tokens):
    token_ids = tokenizer(str(text), add_special_tokens=False)['input_ids']
    if len(token_ids) <= maximum_tokens:
        return str(text)
    beginning_length = int(maximum_tokens * 0.75)
    ending_length = maximum_tokens - beginning_length
    shortened_ids = token_ids[:beginning_length] + token_ids[-ending_length:]
    return tokenizer.decode(shortened_ids, skip_special_tokens=True)

def build_rag_instruction(clause_text, retrieved_examples):
    formatted_examples = []
    for example_number, example in enumerate(retrieved_examples, start=1):
        shortened_example = truncate_clause_for_prompt(example['clause_text'], maximum_tokens=550)
        formatted_examples.append(f"Example {example_number}\nCategory: {example['clause_type']}\nClause: {shortened_example}")
    shortened_query = truncate_clause_for_prompt(clause_text, maximum_tokens=1500)
    examples_text = '\n\n'.join(formatted_examples)
    return f'Classify the new legal clause into exactly ONE of these categories:\n{choices_str}\n\nUse the retrieved labelled training examples as classification examples.\n\nRetrieved training examples:\n{examples_text}\n\nNew clause:\n{shortened_query}\n\nReturn ONLY the category name. Do not explain your answer and do not add punctuation.'

# Load and validate retrieval cache
retrieval_cache = pd.read_csv(RAG_RETRIEVAL_CACHE)
retrieval_cache['row_id'] = retrieval_cache['row_id'].astype(int)
if len(retrieval_cache) != len(rag_test_df):
    raise ValueError('Retrieval cache does not cover the complete test set.')
if set(retrieval_cache['row_id']) != set(rag_test_df['row_id']):
    raise ValueError('Retrieval-cache row IDs do not match the test set.')
retrieval_cache = retrieval_cache.sort_values('row_id').reset_index(drop=True)

# Load resumable prediction checkpoint
result_columns = ['row_id', 'predicted_label_rag', 'raw_output_rag', 'input_tokens']
if os.path.exists(RAG_PARTIAL_PATH):
    completed_rag = pd.read_csv(RAG_PARTIAL_PATH)
    if not set(result_columns).issubset(completed_rag.columns):
        raise ValueError('Existing RAG checkpoint has an incompatible structure.')
    completed_rag['row_id'] = completed_rag['row_id'].astype(int)
    completed_rag = completed_rag.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
else:
    completed_rag = pd.DataFrame(columns=result_columns)
completed_ids = set(completed_rag['row_id']) if len(completed_rag) else set()
remaining_retrievals = retrieval_cache[~retrieval_cache['row_id'].isin(completed_ids)].reset_index(drop=True)
print(f'RAG test inference: {len(completed_rag):,} completed, {len(remaining_retrievals):,} remaining.')

# Run deterministic rag inference
SAVE_EVERY = 20
for start in range(0, len(remaining_retrievals), SAVE_EVERY):
    batch = remaining_retrievals.iloc[start:start + SAVE_EVERY]
    batch_results = []
    for cache_row in batch.itertuples(index=False):
        row_id = int(cache_row.row_id)
        source_row = rag_test_df.iloc[row_id]
        neighbour_indices = json.loads(cache_row.neighbor_indices)
        retrieved_examples = []
        for neighbour_index in neighbour_indices:
            neighbour = rag_metadata_df.iloc[int(neighbour_index)]
            retrieved_examples.append({'clause_text': str(neighbour['clause_text']), 'clause_type': str(neighbour['clause_type'])})
        instruction = build_rag_instruction(source_row['clause_text'], retrieved_examples)
        prompt = build_prompt(instruction)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
        input_token_count = int(inputs['input_ids'].shape[1])
        try:
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=12, do_sample=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
            raw_output = tokenizer.decode(generated[0, input_token_count:], skip_special_tokens=True).strip()
            predicted_label = parse_prediction(raw_output, choices)
        except RuntimeError as error:
            if 'out of memory' in str(error).lower():
                gc.collect()
                torch.cuda.empty_cache()
                raise RuntimeError('GPU memory exhausted during RAG inference. The existing checkpoint is safe; rerun this cell to resume.') from error
            raise
        batch_results.append({'row_id': row_id, 'predicted_label_rag': predicted_label, 'raw_output_rag': raw_output, 'input_tokens': input_token_count})
    completed_rag = pd.concat([completed_rag, pd.DataFrame(batch_results, columns=result_columns)], ignore_index=True)
    completed_rag['row_id'] = completed_rag['row_id'].astype(int)
    completed_rag = completed_rag.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
    completed_rag.to_csv(RAG_PARTIAL_PATH, index=False)
    if len(completed_rag) % 100 == 0 or len(completed_rag) == len(rag_test_df):
        print(f'RAG: {len(completed_rag):,}/{len(rag_test_df):,}')
    gc.collect()
    torch.cuda.empty_cache()

# Final checkpoint integrity
if len(completed_rag) != len(rag_test_df):
    raise RuntimeError('RAG inference is incomplete.')
if set(completed_rag['row_id']) != set(rag_test_df['row_id']):
    raise RuntimeError('RAG prediction row-ID check failed.')
print('RAG inference complete. Proceed to Cell 3D.')


### 3D. Evaluate RAG and retrieval robustness


In [ ]:
# RAG evaluation

# Output paths
RAG_VERSIONED_FINAL = os.path.join(DATA_PATH, 'results_rag_v2_clean_final.csv')
RAG_CANONICAL_FINAL = os.path.join(DATA_PATH, 'results_rag_final.csv')
RAG_OLD_BACKUP = os.path.join(DATA_PATH, 'results_rag_pre_v2_backup.csv')
RAG_REPORT_PATH = os.path.join(DATA_PATH, 'results_rag_v2_clean_report.txt')
RAG_METADATA_OUTPUT = os.path.join(DATA_PATH, 'results_rag_v2_clean_metadata.json')
RAG_SENSITIVITY_PATH = os.path.join(DATA_PATH, 'results_rag_v2_clean_sensitivity.csv')

# Load completed predictions
completed_rag = pd.read_csv(RAG_PARTIAL_PATH)
completed_rag['row_id'] = completed_rag['row_id'].astype(int)
completed_rag = completed_rag.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
if len(completed_rag) != len(test_df):
    raise ValueError(f'RAG prediction checkpoint is incomplete: {len(completed_rag):,}/{len(test_df):,}')
if completed_rag['row_id'].duplicated().any():
    raise ValueError('Duplicate row IDs found in RAG predictions.')

# Load and validate retrieval information
retrieval_cache = pd.read_csv(RAG_RETRIEVAL_CACHE)
retrieval_cache['row_id'] = retrieval_cache['row_id'].astype(int)

def convert_boolean_column(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    converted = series.astype(str).str.strip().str.casefold().map({'true': True, 'false': False})
    if converted.isna().any():
        raise ValueError(f'Invalid Boolean values in {series.name}.')
    return converted
for boolean_column in ['correct_label_in_top3', 'exact_text_match']:
    retrieval_cache[boolean_column] = convert_boolean_column(retrieval_cache[boolean_column])
if len(retrieval_cache) != len(test_df):
    raise ValueError('Retrieval cache is incomplete.')

# Merge source, prediction and retrieval records
rag_results = test_df.reset_index(drop=True).copy()
rag_results.insert(0, 'row_id', range(len(rag_results)))
rag_results = rag_results.merge(completed_rag[['row_id', 'predicted_label_rag', 'raw_output_rag', 'input_tokens']], on='row_id', how='left', validate='one_to_one')
rag_results = rag_results.merge(retrieval_cache[['row_id', 'top1_retrieved_label', 'correct_label_in_top3', 'exact_text_match', 'neighbor_indices', 'neighbor_labels', 'neighbor_distances']], on='row_id', how='left', validate='one_to_one')
required_result_columns = ['predicted_label_rag', 'raw_output_rag', 'correct_label_in_top3', 'exact_text_match']
if rag_results[required_result_columns].isna().any().any():
    raise ValueError('Missing values found after merging RAG results.')
invalid_predictions = sorted(set(rag_results['predicted_label_rag']) - set(choices) - {'Unknown'})
if invalid_predictions:
    raise ValueError(f'Invalid prediction labels found: {invalid_predictions}')

# Metric helper
def calculate_rag_metrics(dataframe, subset_name):
    y_true = dataframe['clause_type']
    y_pred = dataframe['predicted_label_rag']
    return {'subset': subset_name, 'clauses': int(len(dataframe)), 'accuracy': float(accuracy_score(y_true, y_pred)), 'macro_precision': float(precision_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_recall': float(recall_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_f1': float(f1_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'weighted_f1': float(f1_score(y_true, y_pred, labels=choices, average='weighted', zero_division=0)), 'unknown_count': int(y_pred.eq('Unknown').sum()), 'unknown_rate': float(y_pred.eq('Unknown').mean())}

# Primary and robustness metrics
non_exact_results = rag_results.loc[~rag_results['exact_text_match']].copy()
exact_match_results = rag_results.loc[rag_results['exact_text_match']].copy()
primary_metrics = calculate_rag_metrics(rag_results, 'All test clauses')
non_exact_metrics = calculate_rag_metrics(non_exact_results, 'Excluding exact-text matches')
exact_match_metrics = calculate_rag_metrics(exact_match_results, 'Exact-text matches only')
sensitivity_table = pd.DataFrame([primary_metrics, non_exact_metrics, exact_match_metrics])
sensitivity_table.to_csv(RAG_SENSITIVITY_PATH, index=False)

# Classification report
classification_results = classification_report(rag_results['clause_type'], rag_results['predicted_label_rag'], labels=choices, zero_division=0)

# Retrieval-conditioned performance
rag_results['prediction_correct'] = rag_results['clause_type'] == rag_results['predicted_label_rag']
rag_results['is_unknown'] = rag_results['predicted_label_rag'] == 'Unknown'
conditioned_results = rag_results.groupby('correct_label_in_top3', dropna=False).agg(clauses=('row_id', 'size'), accuracy=('prediction_correct', 'mean'), unknown_rate=('is_unknown', 'mean')).reset_index()
conditioned_results['retrieval_condition'] = conditioned_results['correct_label_in_top3'].map({True: 'Correct label retrieved', False: 'Correct label not retrieved'})
conditioned_results = conditioned_results[['retrieval_condition', 'clauses', 'accuracy', 'unknown_rate']]

# Preserve the old canonical rag result
if os.path.exists(RAG_CANONICAL_FINAL) and (not os.path.exists(RAG_OLD_BACKUP)):
    shutil.copy2(RAG_CANONICAL_FINAL, RAG_OLD_BACKUP)

# Save final prediction files
rag_results.to_csv(RAG_VERSIONED_FINAL, index=False)
rag_results.to_csv(RAG_CANONICAL_FINAL, index=False)

# Save report and metadata
with open(RAG_REPORT_PATH, 'w', encoding='utf-8') as report_file:
    report_file.write('RAG TEST RESULTS\n' + '=' * 70 + '\n' + classification_results + '\n\nROBUSTNESS ANALYSIS\n' + '=' * 70 + '\n' + sensitivity_table.to_string(index=False) + '\n\nRETRIEVAL-CONDITIONED RESULTS\n' + '=' * 70 + '\n' + conditioned_results.to_string(index=False))
run_metadata = {'experiment': 'RAG', 'version': 'v2_clean', 'model': MODEL_ID, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'retrieval_k': int(RAG_K), 'distance_metric': 'L2', 'normalized_embeddings': bool(RAG_NORMALIZED), 'indexed_split': 'train', 'test_examples': int(len(rag_results)), 'exact_text_matches': int(rag_results['exact_text_match'].sum()), 'primary_metrics': primary_metrics, 'non_exact_match_metrics': non_exact_metrics, 'completed_at_utc': datetime.now(timezone.utc).isoformat()}
with open(RAG_METADATA_OUTPUT, 'w', encoding='utf-8') as metadata_file:
    json.dump(run_metadata, metadata_file, indent=2)

# Display final results
print(f'Total predictions:       {len(rag_results):,}')
print(f"Accuracy:                {primary_metrics['accuracy']:.4f}")
print(f"Macro precision:         {primary_metrics['macro_precision']:.4f}")
print(f"Macro recall:            {primary_metrics['macro_recall']:.4f}")
print(f"Macro F1:                {primary_metrics['macro_f1']:.4f}")
print(f"Weighted F1:             {primary_metrics['weighted_f1']:.4f}")
print(f"Unknown outputs:         {primary_metrics['unknown_count']:,}")
print(f"Unknown output rate:     {primary_metrics['unknown_rate']:.2%}")
print('\nClassification report:\n')
print(classification_results)
print('\nRobustness analysis:')
display(sensitivity_table.round(4))
print('\nPerformance by retrieval success:')
display(conditioned_results.round(4))
print('\nMost frequent Unknown outputs:')
unknown_outputs = rag_results.loc[rag_results['predicted_label_rag'].eq('Unknown'), 'raw_output_rag'].fillna('<EMPTY>').value_counts().head(20)
if len(unknown_outputs):
    print(unknown_outputs.to_string())
else:
    print('None')
print('\nFiles saved:')
print(f'  Versioned results: {RAG_VERSIONED_FINAL}')
print(f'  Canonical results: {RAG_CANONICAL_FINAL}')
print(f'  Resume checkpoint: {RAG_PARTIAL_PATH}')
print(f'  Sensitivity table: {RAG_SENSITIVITY_PATH}')
print(f'  Run metadata:      {RAG_METADATA_OUTPUT}')


## 4. Hybrid PEFT + RAG

Load the frozen LoRA adapter together with the same cached top-3 retrievals, run hybrid inference, and compare its results with the other neural methods.


### 4A. Load the hybrid model and cached retrievals


In [ ]:
# Hybrid model setup
from peft import PeftModel

# Paths
HYBRID_ADAPTER_DIR = os.path.join(DATA_PATH, 'mistral_legal_adapter_v2_clean')
HYBRID_ADAPTER_CONFIG = os.path.join(HYBRID_ADAPTER_DIR, 'adapter_config.json')
HYBRID_COMPLETE_MARKER = os.path.join(HYBRID_ADAPTER_DIR, 'training_complete.txt')
RAG_METADATA_PATH = os.path.join(DATA_PATH, 'rag_metadata.pkl')
RAG_RETRIEVAL_CACHE = os.path.join(DATA_PATH, 'rag_v2_clean_test_retrievals.csv')

# Verify required assets
required_files = [HYBRID_ADAPTER_CONFIG, HYBRID_COMPLETE_MARKER, RAG_METADATA_PATH, RAG_RETRIEVAL_CACHE]
missing_files = [path for path in required_files if not os.path.exists(path)]
if missing_files:
    raise FileNotFoundError('Required Hybrid assets are missing:\n' + '\n'.join(missing_files))

# Audit adapter configuration
with open(HYBRID_ADAPTER_CONFIG, 'r', encoding='utf-8') as adapter_file:
    adapter_configuration = json.load(adapter_file)
if adapter_configuration.get('r') != 16:
    raise ValueError('Unexpected LoRA rank in adapter configuration.')
if adapter_configuration.get('lora_alpha') != 32:
    raise ValueError('Unexpected LoRA alpha in adapter configuration.')
expected_target_modules = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
actual_target_modules = set(adapter_configuration.get('target_modules', []))
if actual_target_modules != expected_target_modules:
    raise ValueError(f'Unexpected LoRA target modules: {sorted(actual_target_modules)}')

# Reload rag metadata and cached retrievals
with open(RAG_METADATA_PATH, 'rb') as metadata_file:
    rag_metadata = pickle.load(metadata_file)
rag_metadata_df = pd.DataFrame(rag_metadata).reset_index(drop=True)
retrieval_cache = pd.read_csv(RAG_RETRIEVAL_CACHE)
retrieval_cache['row_id'] = retrieval_cache['row_id'].astype(int)
retrieval_cache = retrieval_cache.sort_values('row_id').reset_index(drop=True)
if len(rag_metadata_df) != len(train_df):
    raise ValueError('RAG metadata no longer matches the training set.')
if len(retrieval_cache) != len(test_df):
    raise ValueError('Retrieval cache does not cover the complete test set.')
if set(retrieval_cache['row_id']) != set(range(len(test_df))):
    raise ValueError('Retrieval-cache row IDs do not match the test set.')
for encoded_indices in retrieval_cache['neighbor_indices']:
    indices = json.loads(encoded_indices)
    if len(indices) != 3:
        raise ValueError('Every test clause must have exactly three retrieved examples.')
    if any((index < 0 or index >= len(rag_metadata_df) for index in indices)):
        raise ValueError('Invalid metadata index in retrieval cache.')

# Prepare stable test rows
hybrid_test_df = test_df.reset_index(drop=True).copy()
hybrid_test_df.insert(0, 'row_id', range(len(hybrid_test_df)))

# Remove the experiment 3 base model
for variable_name in ['model', 'base_model', 'trainer', 'embedding_model']:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

# Load clean base model and corrected adapter
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, HYBRID_ADAPTER_DIR, is_trainable=False)
if not hasattr(model, 'peft_config') or len(model.peft_config) != 1:
    raise RuntimeError('Expected exactly one PEFT adapter.')
trainable_parameter_count = sum((parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
if trainable_parameter_count != 0:
    raise RuntimeError('The Hybrid evaluation model unexpectedly contains trainable parameters.')
model.eval()
model.config.use_cache = True
tokenizer.padding_side = 'left'
tokenizer.truncation_side = 'left'

# Conservative parser
def parse_prediction(raw_output, choices):
    if raw_output is None or pd.isna(raw_output):
        return 'Unknown'
    text = str(raw_output).strip()
    if not text:
        return 'Unknown'
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        category_match = re.fullmatch('(?i)category\\s*:\\s*(.+)', candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub('[.,;:!?]+$', '', candidate).strip()
        candidate = re.sub('^[\\s`*_"\\\'“”‘’]+|[\\s`*_"\\\'“”‘’]+$', '', candidate).strip()
        for choice in choices:
            if candidate.casefold() == choice.casefold():
                return choice
    return 'Unknown'

# Identical rag prompt construction
def truncate_clause_for_prompt(text, maximum_tokens):
    token_ids = tokenizer(str(text), add_special_tokens=False)['input_ids']
    if len(token_ids) <= maximum_tokens:
        return str(text)
    beginning_length = int(maximum_tokens * 0.75)
    ending_length = maximum_tokens - beginning_length
    shortened_ids = token_ids[:beginning_length] + token_ids[-ending_length:]
    return tokenizer.decode(shortened_ids, skip_special_tokens=True)

def build_rag_instruction(clause_text, retrieved_examples):
    formatted_examples = []
    for example_number, example in enumerate(retrieved_examples, start=1):
        shortened_example = truncate_clause_for_prompt(example['clause_text'], maximum_tokens=550)
        formatted_examples.append(f"Example {example_number}\nCategory: {example['clause_type']}\nClause: {shortened_example}")
    shortened_query = truncate_clause_for_prompt(clause_text, maximum_tokens=1500)
    examples_text = '\n\n'.join(formatted_examples)
    return f'Classify the new legal clause into exactly ONE of these categories:\n{choices_str}\n\nUse the retrieved labelled training examples as classification examples.\n\nRetrieved training examples:\n{examples_text}\n\nNew clause:\n{shortened_query}\n\nReturn ONLY the category name. Do not explain your answer and do not add punctuation.'
print(f'Base model:             {MODEL_ID}')
print(f'Adapter:                {HYBRID_ADAPTER_DIR}')
print(f'LoRA rank:              16')
print(f'LoRA alpha:             32')
print(f'Loaded adapters:        {len(model.peft_config)}')
print(f'Trainable parameters:   {trainable_parameter_count:,}')
print(f'Cached test retrievals: {len(retrieval_cache):,}')
print(f'Retrieved examples:     3 per test clause')


### 4B. Run hybrid inference


In [ ]:
# Hybrid inference
HYBRID_PARTIAL_PATH = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_partial.csv')
result_columns = ['row_id', 'predicted_label_hybrid', 'raw_output_hybrid', 'input_tokens']

# Load an existing checkpoint if available
if os.path.exists(HYBRID_PARTIAL_PATH):
    completed_hybrid = pd.read_csv(HYBRID_PARTIAL_PATH)
    if not set(result_columns).issubset(completed_hybrid.columns):
        raise ValueError('Existing Hybrid checkpoint has an incompatible structure.')
    completed_hybrid['row_id'] = completed_hybrid['row_id'].astype(int)
    completed_hybrid = completed_hybrid.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
else:
    completed_hybrid = pd.DataFrame(columns=result_columns)
completed_ids = set(completed_hybrid['row_id']) if len(completed_hybrid) else set()
remaining_retrievals = retrieval_cache[~retrieval_cache['row_id'].isin(completed_ids)].sort_values('row_id').reset_index(drop=True)
print(f'Hybrid test inference: {len(completed_hybrid):,} completed, {len(remaining_retrievals):,} remaining.')

# Run deterministic inference
SAVE_EVERY = 20
for start in range(0, len(remaining_retrievals), SAVE_EVERY):
    batch = remaining_retrievals.iloc[start:start + SAVE_EVERY]
    batch_results = []
    for cache_row in batch.itertuples(index=False):
        row_id = int(cache_row.row_id)
        source_row = hybrid_test_df.iloc[row_id]
        neighbour_indices = json.loads(cache_row.neighbor_indices)
        retrieved_examples = []
        for neighbour_index in neighbour_indices:
            neighbour = rag_metadata_df.iloc[int(neighbour_index)]
            retrieved_examples.append({'clause_text': str(neighbour['clause_text']), 'clause_type': str(neighbour['clause_type'])})
        instruction = build_rag_instruction(source_row['clause_text'], retrieved_examples)
        prompt = build_prompt(instruction)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
        input_token_count = int(inputs['input_ids'].shape[1])
        try:
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=12, do_sample=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
            raw_output = tokenizer.decode(generated[0, input_token_count:], skip_special_tokens=True).strip()
            predicted_label = parse_prediction(raw_output, choices)
        except RuntimeError as error:
            if 'out of memory' in str(error).lower():
                gc.collect()
                torch.cuda.empty_cache()
                raise RuntimeError('GPU memory exhausted during Hybrid inference. The saved checkpoint is safe; rerun Cell 4B to resume.') from error
            raise
        batch_results.append({'row_id': row_id, 'predicted_label_hybrid': predicted_label, 'raw_output_hybrid': raw_output, 'input_tokens': input_token_count})
    completed_hybrid = pd.concat([completed_hybrid, pd.DataFrame(batch_results, columns=result_columns)], ignore_index=True)
    completed_hybrid['row_id'] = completed_hybrid['row_id'].astype(int)
    completed_hybrid = completed_hybrid.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
    completed_hybrid.to_csv(HYBRID_PARTIAL_PATH, index=False)
    if len(completed_hybrid) % 100 == 0 or len(completed_hybrid) == len(hybrid_test_df):
        print(f'Hybrid: {len(completed_hybrid):,}/{len(hybrid_test_df):,}')
    gc.collect()
    torch.cuda.empty_cache()

# Check completion
if len(completed_hybrid) != len(hybrid_test_df):
    raise RuntimeError('Hybrid inference is incomplete.')
if set(completed_hybrid['row_id']) != set(hybrid_test_df['row_id']):
    raise RuntimeError('Hybrid prediction row-ID check failed.')
print('Hybrid inference complete. Proceed to Cell 4C.')


### 4C. Evaluate hybrid performance and robustness


In [ ]:
# Hybrid evaluation

# Output paths
HYBRID_VERSIONED_FINAL = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_final.csv')
HYBRID_CANONICAL_FINAL = os.path.join(DATA_PATH, 'results_rag_peft_hybrid.csv')
HYBRID_OLD_BACKUP = os.path.join(DATA_PATH, 'results_rag_peft_hybrid_pre_v2_backup.csv')
HYBRID_REPORT_PATH = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_report.txt')
HYBRID_METADATA_PATH = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_metadata.json')
HYBRID_SENSITIVITY_PATH = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_sensitivity.csv')
HYBRID_PAIRED_PATH = os.path.join(DATA_PATH, 'results_hybrid_vs_rag_paired.csv')
RAG_REFERENCE_PATH = os.path.join(DATA_PATH, 'results_rag_v2_clean_final.csv')

# Load completed predictions
completed_hybrid = pd.read_csv(HYBRID_PARTIAL_PATH)
completed_hybrid['row_id'] = completed_hybrid['row_id'].astype(int)
completed_hybrid = completed_hybrid.drop_duplicates('row_id', keep='last').sort_values('row_id').reset_index(drop=True)
if len(completed_hybrid) != len(test_df):
    raise ValueError('Hybrid prediction checkpoint is incomplete.')

# Prepare retrieval information
retrieval_cache = pd.read_csv(RAG_RETRIEVAL_CACHE)
retrieval_cache['row_id'] = retrieval_cache['row_id'].astype(int)

def convert_boolean_column(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    converted = series.astype(str).str.strip().str.casefold().map({'true': True, 'false': False})
    if converted.isna().any():
        raise ValueError(f'Invalid Boolean values in {series.name}.')
    return converted
for boolean_column in ['correct_label_in_top3', 'exact_text_match']:
    retrieval_cache[boolean_column] = convert_boolean_column(retrieval_cache[boolean_column])

# Merge test data, predictions and retrieval diagnostics
hybrid_results = test_df.reset_index(drop=True).copy()
hybrid_results.insert(0, 'row_id', range(len(hybrid_results)))
hybrid_results = hybrid_results.merge(completed_hybrid[['row_id', 'predicted_label_hybrid', 'raw_output_hybrid', 'input_tokens']], on='row_id', how='left', validate='one_to_one')
hybrid_results = hybrid_results.merge(retrieval_cache[['row_id', 'top1_retrieved_label', 'correct_label_in_top3', 'exact_text_match', 'neighbor_indices', 'neighbor_labels', 'neighbor_distances']], on='row_id', how='left', validate='one_to_one')
if hybrid_results[['predicted_label_hybrid', 'correct_label_in_top3', 'exact_text_match']].isna().any().any():
    raise ValueError('Missing values found after merging Hybrid results.')
invalid_predictions = sorted(set(hybrid_results['predicted_label_hybrid']) - set(choices) - {'Unknown'})
if invalid_predictions:
    raise ValueError(f'Invalid Hybrid prediction labels: {invalid_predictions}')

# Metric helper
def calculate_hybrid_metrics(dataframe, subset_name):
    y_true = dataframe['clause_type']
    y_pred = dataframe['predicted_label_hybrid']
    return {'subset': subset_name, 'clauses': int(len(dataframe)), 'accuracy': float(accuracy_score(y_true, y_pred)), 'macro_precision': float(precision_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_recall': float(recall_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_f1': float(f1_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'weighted_f1': float(f1_score(y_true, y_pred, labels=choices, average='weighted', zero_division=0)), 'unknown_count': int(y_pred.eq('Unknown').sum()), 'unknown_rate': float(y_pred.eq('Unknown').mean())}

# Primary and exact-match robustness results
non_exact_results = hybrid_results.loc[~hybrid_results['exact_text_match']].copy()
exact_match_results = hybrid_results.loc[hybrid_results['exact_text_match']].copy()
primary_metrics = calculate_hybrid_metrics(hybrid_results, 'All test clauses')
non_exact_metrics = calculate_hybrid_metrics(non_exact_results, 'Excluding exact-text matches')
exact_match_metrics = calculate_hybrid_metrics(exact_match_results, 'Exact-text matches only')
sensitivity_table = pd.DataFrame([primary_metrics, non_exact_metrics, exact_match_metrics])
sensitivity_table.to_csv(HYBRID_SENSITIVITY_PATH, index=False)

# Classification report
classification_results = classification_report(hybrid_results['clause_type'], hybrid_results['predicted_label_hybrid'], labels=choices, zero_division=0)

# Performance conditional on retrieval success
hybrid_results['prediction_correct'] = hybrid_results['clause_type'] == hybrid_results['predicted_label_hybrid']
hybrid_results['is_unknown'] = hybrid_results['predicted_label_hybrid'] == 'Unknown'
conditioned_results = hybrid_results.groupby('correct_label_in_top3', dropna=False).agg(clauses=('row_id', 'size'), accuracy=('prediction_correct', 'mean'), unknown_rate=('is_unknown', 'mean')).reset_index()
conditioned_results['retrieval_condition'] = conditioned_results['correct_label_in_top3'].map({True: 'Correct label retrieved', False: 'Correct label not retrieved'})
conditioned_results = conditioned_results[['retrieval_condition', 'clauses', 'accuracy', 'unknown_rate']]

# Paired hybrid-versus-rag comparison
if not os.path.exists(RAG_REFERENCE_PATH):
    raise FileNotFoundError(f'Corrected RAG result file not found:\n{RAG_REFERENCE_PATH}')
rag_reference = pd.read_csv(RAG_REFERENCE_PATH)
if 'row_id' not in rag_reference.columns:
    raise ValueError('RAG reference file has no row_id column.')
paired_comparison = hybrid_results[['row_id', 'clause_type', 'predicted_label_hybrid']].merge(rag_reference[['row_id', 'predicted_label_rag']], on='row_id', how='inner', validate='one_to_one')
if len(paired_comparison) != len(test_df):
    raise ValueError('Hybrid/RAG paired comparison is incomplete.')
paired_comparison['rag_correct'] = paired_comparison['clause_type'] == paired_comparison['predicted_label_rag']
paired_comparison['hybrid_correct'] = paired_comparison['clause_type'] == paired_comparison['predicted_label_hybrid']

def paired_outcome(row):
    if row['rag_correct'] and row['hybrid_correct']:
        return 'Both correct'
    if row['rag_correct']:
        return 'RAG only correct'
    if row['hybrid_correct']:
        return 'Hybrid only correct'
    return 'Both incorrect'
paired_comparison['outcome'] = paired_comparison.apply(paired_outcome, axis=1)
paired_comparison.to_csv(HYBRID_PAIRED_PATH, index=False)
paired_summary = paired_comparison['outcome'].value_counts().reindex(['Both correct', 'RAG only correct', 'Hybrid only correct', 'Both incorrect'], fill_value=0).rename_axis('paired_outcome').reset_index(name='clauses')

# Preserve old canonical hybrid result
if os.path.exists(HYBRID_CANONICAL_FINAL) and (not os.path.exists(HYBRID_OLD_BACKUP)):
    shutil.copy2(HYBRID_CANONICAL_FINAL, HYBRID_OLD_BACKUP)

# Save final outputs
hybrid_results.to_csv(HYBRID_VERSIONED_FINAL, index=False)
hybrid_results.to_csv(HYBRID_CANONICAL_FINAL, index=False)
with open(HYBRID_REPORT_PATH, 'w', encoding='utf-8') as report_file:
    report_file.write('HYBRID PEFT + RAG TEST RESULTS\n' + '=' * 70 + '\n' + classification_results + '\n\nROBUSTNESS ANALYSIS\n' + '=' * 70 + '\n' + sensitivity_table.to_string(index=False) + '\n\nRETRIEVAL-CONDITIONED RESULTS\n' + '=' * 70 + '\n' + conditioned_results.to_string(index=False) + '\n\nPAIRED HYBRID-VERSUS-RAG RESULTS\n' + '=' * 70 + '\n' + paired_summary.to_string(index=False))
run_metadata = {'experiment': 'PEFT + RAG Hybrid', 'version': 'v2_clean', 'model': MODEL_ID, 'adapter': HYBRID_ADAPTER_DIR, 'retrieval_k': 3, 'retrieval_cache': RAG_RETRIEVAL_CACHE, 'test_examples': int(len(hybrid_results)), 'exact_text_matches': int(hybrid_results['exact_text_match'].sum()), 'primary_metrics': primary_metrics, 'non_exact_match_metrics': non_exact_metrics, 'paired_comparison': {row['paired_outcome']: int(row['clauses']) for _, row in paired_summary.iterrows()}, 'completed_at_utc': datetime.now(timezone.utc).isoformat()}
with open(HYBRID_METADATA_PATH, 'w', encoding='utf-8') as metadata_file:
    json.dump(run_metadata, metadata_file, indent=2)

# Display results
print(f'Total predictions:       {len(hybrid_results):,}')
print(f"Accuracy:                {primary_metrics['accuracy']:.4f}")
print(f"Macro precision:         {primary_metrics['macro_precision']:.4f}")
print(f"Macro recall:            {primary_metrics['macro_recall']:.4f}")
print(f"Macro F1:                {primary_metrics['macro_f1']:.4f}")
print(f"Weighted F1:             {primary_metrics['weighted_f1']:.4f}")
print(f"Unknown outputs:         {primary_metrics['unknown_count']:,}")
print(f"Unknown output rate:     {primary_metrics['unknown_rate']:.2%}")
print('\nClassification report:\n')
print(classification_results)
print('\nRobustness analysis:')
display(sensitivity_table.round(4))
print('\nPerformance by retrieval success:')
display(conditioned_results.round(4))
print('\nPaired comparison with RAG:')
display(paired_summary)
rag_accuracy = paired_comparison['rag_correct'].mean()
hybrid_accuracy = paired_comparison['hybrid_correct'].mean()
print(f'\nPaired accuracy difference (Hybrid - RAG): {hybrid_accuracy - rag_accuracy:+.4f}')
print('\nMost frequent Unknown outputs:')
unknown_outputs = hybrid_results.loc[hybrid_results['predicted_label_hybrid'].eq('Unknown'), 'raw_output_hybrid'].fillna('<EMPTY>').value_counts().head(20)
if len(unknown_outputs):
    print(unknown_outputs.to_string())
else:
    print('None')
print('\nFiles saved:')
print(f'  Versioned results: {HYBRID_VERSIONED_FINAL}')
print(f'  Canonical results: {HYBRID_CANONICAL_FINAL}')
print(f'  Resume checkpoint: {HYBRID_PARTIAL_PATH}')
print(f'  Sensitivity table: {HYBRID_SENSITIVITY_PATH}')
print(f'  Paired comparison: {HYBRID_PAIRED_PATH}')
print(f'  Run metadata:      {HYBRID_METADATA_PATH}')


## 5. Authoritative five-method analysis

Recover row-level predictions for the classical baseline, align all five methods, compute paired uncertainty and significance tests, regenerate final figures, and archive the authoritative artifacts.


### 5A. Reproduce the TF-IDF baseline and recover row-level predictions

The archived test accuracy, Macro F1, and Weighted F1 must reproduce exactly. Minor train/validation drift up to 0.005 is documented and accepted because it does not affect the frozen test predictions used in the paired analysis.


In [ ]:
# Baseline reproduction
import platform
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Configuration and paths
DATA_PATH = '/content/drive/MyDrive/Thesis1_Project'
RANDOM_SEED = 42
TRAIN_PATH = os.path.join(DATA_PATH, 'train.csv')
VALIDATION_PATH = os.path.join(DATA_PATH, 'val.csv')
TEST_PATH = os.path.join(DATA_PATH, 'test.csv')
ARCHIVED_METRICS_PATH = os.path.join(DATA_PATH, 'baseline_results.json')
BASELINE_VALIDATION_RESULTS = os.path.join(DATA_PATH, 'results_baseline_reproduced_validation.csv')
BASELINE_VERSIONED_RESULTS = os.path.join(DATA_PATH, 'results_baseline_reproduced_final.csv')
BASELINE_CANONICAL_RESULTS = os.path.join(DATA_PATH, 'results_baseline_final.csv')
BASELINE_MODEL_PATH = os.path.join(DATA_PATH, 'baseline_tfidf_logistic_model.joblib')
BASELINE_METADATA_PATH = os.path.join(DATA_PATH, 'results_baseline_reproduced_metadata.json')
BASELINE_REPORT_PATH = os.path.join(DATA_PATH, 'results_baseline_reproduced_report.txt')

# Load frozen data splits and archived metrics
train_df = pd.read_csv(TRAIN_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_PATH).reset_index(drop=True)
expected_split_sizes = {'training': 8257, 'validation': 833, 'test': 1850}
observed_split_sizes = {'training': len(train_df), 'validation': len(validation_df), 'test': len(test_df)}
if observed_split_sizes != expected_split_sizes:
    raise ValueError(f'Dataset split sizes differ from the original baseline experiment.\nExpected: {expected_split_sizes}\nObserved: {observed_split_sizes}')
for split_name, dataframe in [('training', train_df), ('validation', validation_df), ('test', test_df)]:
    missing_columns = {'clause_text', 'clause_type', 'contract_title'} - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f'{split_name.title()} data is missing: {sorted(missing_columns)}')
    if dataframe[['clause_text', 'clause_type']].isna().any().any():
        raise ValueError(f'Missing clause text or labels in {split_name} data.')
with open(ARCHIVED_METRICS_PATH, 'r', encoding='utf-8') as metrics_file:
    archived_metrics = json.load(metrics_file)

# Reproduce the exact original feature pipeline
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
training_features = tfidf_vectorizer.fit_transform(train_df['clause_text'])
validation_features = tfidf_vectorizer.transform(validation_df['clause_text'])
test_features = tfidf_vectorizer.transform(test_df['clause_text'])

# Reproduce the exact original classifier
baseline_classifier = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED)
baseline_classifier.fit(training_features, train_df['clause_type'])
training_predictions = baseline_classifier.predict(training_features)
validation_predictions = baseline_classifier.predict(validation_features)
test_predictions = baseline_classifier.predict(test_features)

# Recalculate original metrics
reproduced_metrics = {'train_accuracy': float(accuracy_score(train_df['clause_type'], training_predictions)), 'val_accuracy': float(accuracy_score(validation_df['clause_type'], validation_predictions)), 'test_accuracy': float(accuracy_score(test_df['clause_type'], test_predictions)), 'val_macro_f1': float(f1_score(validation_df['clause_type'], validation_predictions, average='macro', zero_division=0)), 'test_macro_f1': float(f1_score(test_df['clause_type'], test_predictions, average='macro', zero_division=0)), 'val_weighted_f1': float(f1_score(validation_df['clause_type'], validation_predictions, average='weighted', zero_division=0)), 'test_weighted_f1': float(f1_score(test_df['clause_type'], test_predictions, average='weighted', zero_division=0))}

# Verify reproduction before saving anything
verification_records = []
for metric_name, reproduced_value in reproduced_metrics.items():
    if metric_name not in archived_metrics:
        raise KeyError(f'Archived baseline is missing {metric_name}.')
    archived_value = float(archived_metrics[metric_name])
    absolute_difference = abs(reproduced_value - archived_value)
    matches = bool(np.isclose(reproduced_value, archived_value, rtol=0, atol=1e-12))
    verification_records.append({'metric': metric_name, 'archived_value': archived_value, 'reproduced_value': reproduced_value, 'absolute_difference': absolute_difference, 'exact_match_within_tolerance': matches})
verification_table = pd.DataFrame(verification_records)
display(verification_table)

# Require exact agreement on every archived test metric
required_test_metrics = ['test_accuracy', 'test_macro_f1', 'test_weighted_f1']
test_verification = verification_table.loc[verification_table['metric'].isin(required_test_metrics)].copy()
if len(test_verification) != len(required_test_metrics):
    raise RuntimeError('The test verification table is incomplete.')
if not test_verification['exact_match_within_tolerance'].all():
    raise RuntimeError('At least one archived test metric does not match. Baseline predictions will not be saved.')

# Record non-test numerical drift explicitly
non_test_drift = verification_table.loc[~verification_table['metric'].isin(required_test_metrics)].copy()
maximum_non_test_difference = float(non_test_drift['absolute_difference'].max())
if maximum_non_test_difference > 0.005:
    raise RuntimeError('Training/validation drift exceeded the documented 0.005 tolerance; no baseline artifacts were saved.')
print('All archived test metrics: Exact match')
print(f'Maximum training/validation metric drift: {maximum_non_test_difference:.6f}')
print('Status: Exact test-metric reproduction with minor non-test numerical drift.')

# Build row-level outputs
validation_results = validation_df.reset_index(drop=True).copy()
validation_results.insert(0, 'row_id', range(len(validation_results)))
validation_results['predicted_label_baseline'] = validation_predictions
test_results = test_df.reset_index(drop=True).copy()
test_results.insert(0, 'row_id', range(len(test_results)))
test_results['predicted_label_baseline'] = test_predictions
if len(test_results) != 1850:
    raise RuntimeError('Baseline test output does not contain 1,850 rows.')
if test_results['predicted_label_baseline'].isna().any():
    raise RuntimeError('Missing baseline test predictions.')
invalid_labels = sorted(set(test_results['predicted_label_baseline']) - set(test_results['clause_type']))
if invalid_labels:
    raise RuntimeError(f'Unexpected baseline labels: {invalid_labels}')

# Calculate additional test metrics
test_macro_precision = precision_score(test_results['clause_type'], test_results['predicted_label_baseline'], average='macro', zero_division=0)
test_macro_recall = recall_score(test_results['clause_type'], test_results['predicted_label_baseline'], average='macro', zero_division=0)
test_report = classification_report(test_results['clause_type'], test_results['predicted_label_baseline'], zero_division=0)

# Save predictions and fitted pipeline
BASELINE_VERIFICATION_PATH = os.path.join(DATA_PATH, 'baseline_reproduction_verification.csv')
verification_table.to_csv(BASELINE_VERIFICATION_PATH, index=False)
validation_results.to_csv(BASELINE_VALIDATION_RESULTS, index=False)
test_results.to_csv(BASELINE_VERSIONED_RESULTS, index=False)
test_results.to_csv(BASELINE_CANONICAL_RESULTS, index=False)
joblib.dump({'tfidf_vectorizer': tfidf_vectorizer, 'logistic_regression': baseline_classifier}, BASELINE_MODEL_PATH)

# Save transparent reproduction metadata
baseline_metadata = {'status': 'exact_test_metric_reproduction_with_minor_train_validation_numerical_drift', 'source_implementation': 'data_prep_final.py', 'random_seed': 42, 'training_examples': int(len(train_df)), 'validation_examples': int(len(validation_df)), 'test_examples': int(len(test_df)), 'tfidf_configuration': {'max_features': 5000, 'ngram_range': [1, 2], 'other_parameters': 'scikit-learn defaults'}, 'logistic_regression_configuration': {'max_iter': 1000, 'class_weight': 'balanced', 'random_state': 42, 'other_parameters': 'scikit-learn defaults'}, 'software': {'python': platform.python_version(), 'scikit_learn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__}, 'archived_metrics': archived_metrics, 'reproduced_metrics': reproduced_metrics, 'verification': verification_table.to_dict(orient='records'), 'test_metric_match': True, 'full_metric_match': False, 'maximum_non_test_absolute_drift': maximum_non_test_difference, 'interpretation': 'All archived test metrics matched exactly. Small training and validation differences are consistent with numerical or library-version variation in logistic-regression optimization.', 'additional_test_metrics': {'macro_precision': float(test_macro_precision), 'macro_recall': float(test_macro_recall)}, 'completed_at_utc': datetime.now(timezone.utc).isoformat()}
with open(BASELINE_METADATA_PATH, 'w', encoding='utf-8') as metadata_file:
    json.dump(baseline_metadata, metadata_file, indent=2)
with open(BASELINE_REPORT_PATH, 'w', encoding='utf-8') as report_file:
    report_file.write('REPRODUCED BASELINE TEST REPORT\n' + '=' * 70 + '\n' + test_report + '\n\nREPRODUCTION STATUS\n' + '=' * 70 + '\n' + 'All archived test metrics matched exactly. Minor numerical drift was observed in the training and validation metrics.')

# Final confirmation
print(f"Test accuracy:        {reproduced_metrics['test_accuracy']:.4f}")
print(f'Macro precision:      {test_macro_precision:.4f}')
print(f'Macro recall:         {test_macro_recall:.4f}')
print(f"Macro F1:             {reproduced_metrics['test_macro_f1']:.4f}")
print(f"Weighted F1:          {reproduced_metrics['test_weighted_f1']:.4f}")
print('Reproduction status: Exact test metrics; minor train/validation numerical drift')
print('\nFiles saved:')
print(f'  Test predictions: {BASELINE_CANONICAL_RESULTS}')
print(f'  Fitted pipeline:  {BASELINE_MODEL_PATH}')
print(f'  Verification:     {BASELINE_VERIFICATION_PATH}')
print(f'  Metadata:         {BASELINE_METADATA_PATH}')


### 5B. Five-method metrics, confidence intervals, and paired tests


In [ ]:
# Five-method statistical analysis
from itertools import combinations
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Self-contained configuration
DATA_PATH = '/content/drive/MyDrive/Thesis1_Project'
choices = ['Parties', 'License Grant', 'Cap On Liability', 'Audit Rights', 'Anti-Assignment', 'Insurance', 'Document Name', 'Agreement Date', 'Governing Law', 'Expiration Date', 'Effective Date', 'Post-Termination Services', 'Revenue/Profit Sharing', 'Minimum Commitment', 'Exclusivity', 'Rofr/Rofo/Rofn', 'Ip Ownership Assignment', 'Non-Transferable License', 'Non-Compete', 'Termination For Convenience']
TEST_PATH = os.path.join(DATA_PATH, 'test.csv')
COMBINED_PREDICTIONS_PATH = os.path.join(DATA_PATH, 'final_all_model_predictions.csv')
FINAL_COMPARISON_PATH = os.path.join(DATA_PATH, 'final_comparison.csv')
BOOTSTRAP_CI_PATH = os.path.join(DATA_PATH, 'final_bootstrap_confidence_intervals.csv')
BOOTSTRAP_SAMPLES_PATH = os.path.join(DATA_PATH, 'final_bootstrap_samples.npz')
MCNEMAR_RESULTS_PATH = os.path.join(DATA_PATH, 'final_pairwise_mcnemar_tests.csv')
MACRO_F1_DIFFERENCE_PATH = os.path.join(DATA_PATH, 'final_pairwise_macro_f1_bootstrap.csv')
method_files = {'TF-IDF + Logistic Regression': {'filename': 'results_baseline_final.csv', 'source_column': 'predicted_label_baseline', 'combined_column': 'predicted_label_baseline'}, 'Zero-shot': {'filename': 'results_prompt_engineering_final.csv', 'source_column': 'predicted_label', 'combined_column': 'predicted_label_zero_shot'}, 'PEFT': {'filename': 'results_peft_final.csv', 'source_column': 'predicted_label_peft', 'combined_column': 'predicted_label_peft'}, 'RAG': {'filename': 'results_rag_final.csv', 'source_column': 'predicted_label_rag', 'combined_column': 'predicted_label_rag'}, 'Hybrid': {'filename': 'results_rag_peft_hybrid.csv', 'source_column': 'predicted_label_hybrid', 'combined_column': 'predicted_label_hybrid'}}
method_order = list(method_files.keys())
prediction_columns = {method_name: specification['combined_column'] for method_name, specification in method_files.items()}

# Rebuild and audit the aligned prediction matrix
test_reference = pd.read_csv(TEST_PATH).reset_index(drop=True)
test_reference.insert(0, 'row_id', range(len(test_reference)))
if len(test_reference) != 1850:
    raise ValueError('Expected 1,850 test rows.')
combined_predictions = test_reference[['row_id', 'clause_text', 'clause_type', 'contract_title']].copy()
audit_records = []
for method_name, specification in method_files.items():
    result_path = os.path.join(DATA_PATH, specification['filename'])
    if not os.path.exists(result_path):
        raise FileNotFoundError(f'{method_name} results missing:\n{result_path}')
    result_df = pd.read_csv(result_path)
    source_column = specification['source_column']
    required_columns = {'row_id', 'clause_text', 'clause_type', source_column}
    missing_columns = required_columns - set(result_df.columns)
    if missing_columns:
        raise ValueError(f'{method_name} is missing columns: {sorted(missing_columns)}')
    result_df['row_id'] = result_df['row_id'].astype(int)
    if result_df['row_id'].duplicated().any():
        raise ValueError(f'{method_name} has duplicate row IDs.')
    result_df = result_df.sort_values('row_id').reset_index(drop=True)
    if len(result_df) != len(test_reference):
        raise ValueError(f'{method_name} has an incorrect number of rows.')
    if result_df['row_id'].tolist() != test_reference['row_id'].tolist():
        raise ValueError(f'{method_name} row-ID alignment failed.')
    if result_df['clause_type'].astype(str).tolist() != test_reference['clause_type'].astype(str).tolist():
        raise ValueError(f'{method_name} true-label alignment failed.')
    if result_df['clause_text'].fillna('').astype(str).tolist() != test_reference['clause_text'].fillna('').astype(str).tolist():
        raise ValueError(f'{method_name} text alignment failed.')
    predictions = result_df[source_column].astype(str)
    invalid_predictions = sorted(set(predictions) - set(choices) - {'Unknown'})
    if invalid_predictions:
        raise ValueError(f'{method_name} contains invalid predictions: {invalid_predictions}')
    combined_column = specification['combined_column']
    combined_predictions[combined_column] = predictions.values
    audit_records.append({'method': method_name, 'rows': int(len(result_df)), 'unknown_count': int(predictions.eq('Unknown').sum()), 'unknown_rate': float(predictions.eq('Unknown').mean()), 'alignment': 'Passed'})
combined_predictions.to_csv(COMBINED_PREDICTIONS_PATH, index=False)
audit_table = pd.DataFrame(audit_records)

# Recalculate all observed metrics from predictions
y_true = combined_predictions['clause_type'].astype(str)
comparison_records = []
for method_name in method_order:
    y_pred = combined_predictions[prediction_columns[method_name]].astype(str)
    comparison_records.append({'method': method_name, 'accuracy': float(accuracy_score(y_true, y_pred)), 'macro_precision': float(precision_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_recall': float(recall_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'macro_f1': float(f1_score(y_true, y_pred, labels=choices, average='macro', zero_division=0)), 'weighted_f1': float(f1_score(y_true, y_pred, labels=choices, average='weighted', zero_division=0)), 'unknown_count': int(y_pred.eq('Unknown').sum()), 'unknown_rate': float(y_pred.eq('Unknown').mean())})
comparison_table = pd.DataFrame(comparison_records)
comparison_table.to_csv(FINAL_COMPARISON_PATH, index=False)

# Encode labels for efficient paired bootstrapping
label_to_id = {label: index for index, label in enumerate(choices)}
UNKNOWN_ID = len(choices)
NUMBER_OF_PREDICTION_LABELS = len(choices) + 1
true_ids = y_true.map(label_to_id).to_numpy(dtype=np.int64)
prediction_ids = {}
for method_name in method_order:
    prediction_ids[method_name] = combined_predictions[prediction_columns[method_name]].map(lambda value: label_to_id[value] if value in label_to_id else UNKNOWN_ID).to_numpy(dtype=np.int64)

def calculate_encoded_metrics(sampled_true, sampled_prediction):
    counts = np.bincount(sampled_true * NUMBER_OF_PREDICTION_LABELS + sampled_prediction, minlength=len(choices) * NUMBER_OF_PREDICTION_LABELS).reshape(len(choices), NUMBER_OF_PREDICTION_LABELS)
    true_positive = np.diag(counts[:, :len(choices)]).astype(float)
    true_support = counts.sum(axis=1).astype(float)
    predicted_support = counts[:, :len(choices)].sum(axis=0).astype(float)
    f1_denominator = true_support + predicted_support
    per_class_f1 = np.divide(2.0 * true_positive, f1_denominator, out=np.zeros_like(true_positive), where=f1_denominator != 0)
    return (float((sampled_true == sampled_prediction).mean()), float(per_class_f1.mean()))

# Run paired bootstrap for all five methods
NUMBER_OF_BOOTSTRAPS = 5000
RANDOM_SEED = 42
random_generator = np.random.default_rng(RANDOM_SEED)
number_of_rows = len(combined_predictions)
bootstrap_samples = {method_name: {'accuracy': np.empty(NUMBER_OF_BOOTSTRAPS), 'macro_f1': np.empty(NUMBER_OF_BOOTSTRAPS)} for method_name in method_order}
for bootstrap_index in range(NUMBER_OF_BOOTSTRAPS):
    sampled_indices = random_generator.integers(0, number_of_rows, size=number_of_rows)
    sampled_true = true_ids[sampled_indices]
    for method_name in method_order:
        bootstrap_accuracy, bootstrap_macro_f1 = calculate_encoded_metrics(sampled_true, prediction_ids[method_name][sampled_indices])
        bootstrap_samples[method_name]['accuracy'][bootstrap_index] = bootstrap_accuracy
        bootstrap_samples[method_name]['macro_f1'][bootstrap_index] = bootstrap_macro_f1

def safe_method_name(method_name):
    return re.sub('[^a-z0-9]+', '_', method_name.casefold()).strip('_')
samples_to_save = {}
for method_name in method_order:
    method_key = safe_method_name(method_name)
    samples_to_save[f'{method_key}_accuracy'] = bootstrap_samples[method_name]['accuracy']
    samples_to_save[f'{method_key}_macro_f1'] = bootstrap_samples[method_name]['macro_f1']
np.savez_compressed(BOOTSTRAP_SAMPLES_PATH, **samples_to_save)

# Calculate individual confidence intervals
confidence_interval_records = []
for method_name in method_order:
    observed = comparison_table.loc[comparison_table['method'].eq(method_name)].iloc[0]
    confidence_interval_records.append({'method': method_name, 'accuracy': float(observed['accuracy']), 'accuracy_ci_lower': float(np.percentile(bootstrap_samples[method_name]['accuracy'], 2.5)), 'accuracy_ci_upper': float(np.percentile(bootstrap_samples[method_name]['accuracy'], 97.5)), 'macro_f1': float(observed['macro_f1']), 'macro_f1_ci_lower': float(np.percentile(bootstrap_samples[method_name]['macro_f1'], 2.5)), 'macro_f1_ci_upper': float(np.percentile(bootstrap_samples[method_name]['macro_f1'], 97.5))})
confidence_interval_table = pd.DataFrame(confidence_interval_records)
confidence_interval_table.to_csv(BOOTSTRAP_CI_PATH, index=False)

# Exact mcnemar tests for all method pairs
correctness = {method_name: (combined_predictions[prediction_columns[method_name]].astype(str) == y_true).to_numpy() for method_name in method_order}

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_maximum = 0.0
    for rank, original_index in enumerate(order):
        corrected = min(1.0, (len(p_values) - rank) * p_values[original_index])
        running_maximum = max(running_maximum, corrected)
        adjusted[original_index] = running_maximum
    return adjusted
mcnemar_records = []
for method_a, method_b in combinations(method_order, 2):
    a_correct = correctness[method_a]
    b_correct = correctness[method_b]
    a_only_correct = int(np.sum(a_correct & ~b_correct))
    b_only_correct = int(np.sum(~a_correct & b_correct))
    discordant_pairs = a_only_correct + b_only_correct
    raw_p_value = 1.0 if discordant_pairs == 0 else float(binomtest(a_only_correct, discordant_pairs, p=0.5, alternative='two-sided').pvalue)
    mcnemar_records.append({'method_a': method_a, 'method_b': method_b, 'accuracy_a': float(a_correct.mean()), 'accuracy_b': float(b_correct.mean()), 'accuracy_difference_b_minus_a': float(b_correct.mean() - a_correct.mean()), 'a_only_correct': a_only_correct, 'b_only_correct': b_only_correct, 'discordant_pairs': discordant_pairs, 'raw_p_value': raw_p_value})
mcnemar_table = pd.DataFrame(mcnemar_records)
mcnemar_table['holm_adjusted_p_value'] = holm_adjust(mcnemar_table['raw_p_value'])
mcnemar_table['significant_after_holm_0.05'] = mcnemar_table['holm_adjusted_p_value'] < 0.05
mcnemar_table.to_csv(MCNEMAR_RESULTS_PATH, index=False)


In [ ]:
# Paired bootstrap intervals for macro f1 differences
macro_difference_records = []
for method_a, method_b in combinations(method_order, 2):
    observed_a = float(comparison_table.loc[comparison_table['method'].eq(method_a), 'macro_f1'].iloc[0])
    observed_b = float(comparison_table.loc[comparison_table['method'].eq(method_b), 'macro_f1'].iloc[0])
    bootstrap_difference = bootstrap_samples[method_b]['macro_f1'] - bootstrap_samples[method_a]['macro_f1']
    confidence_lower = float(np.percentile(bootstrap_difference, 2.5))
    confidence_upper = float(np.percentile(bootstrap_difference, 97.5))
    if confidence_lower > 0:
        conclusion = f'{method_b} higher'
    elif confidence_upper < 0:
        conclusion = f'{method_a} higher'
    else:
        conclusion = 'No clear difference'
    macro_difference_records.append({'method_a': method_a, 'method_b': method_b, 'macro_f1_a': observed_a, 'macro_f1_b': observed_b, 'difference_b_minus_a': observed_b - observed_a, 'difference_ci_lower': confidence_lower, 'difference_ci_upper': confidence_upper, 'bootstrap_probability_b_higher': float(np.mean(bootstrap_difference > 0)), 'interval_conclusion': conclusion})
macro_f1_difference_table = pd.DataFrame(macro_difference_records)
macro_f1_difference_table.to_csv(MACRO_F1_DIFFERENCE_PATH, index=False)

# Extract baseline comparisons
baseline_accuracy_tests = mcnemar_table.loc[mcnemar_table['method_a'].eq('TF-IDF + Logistic Regression')].reset_index(drop=True)
baseline_macro_comparisons = macro_f1_difference_table.loc[macro_f1_difference_table['method_a'].eq('TF-IDF + Logistic Regression')].reset_index(drop=True)

# Display and save final results
print('\nFive-method integrity audit:')
display(audit_table.round(4))
print('\nFinal comparison:')
display(comparison_table.round(4))
print('\n95% bootstrap confidence intervals:')
display(confidence_interval_table.round(4))
display(baseline_accuracy_tests.round(6))
display(baseline_macro_comparisons.round(6))
print('\nFiles updated:')
print(f'  Combined predictions: {COMBINED_PREDICTIONS_PATH}')
print(f'  Final comparison:     {FINAL_COMPARISON_PATH}')
print(f'  Bootstrap intervals:  {BOOTSTRAP_CI_PATH}')
print(f'  McNemar tests:        {MCNEMAR_RESULTS_PATH}')
print(f'  Macro F1 differences: {MACRO_F1_DIFFERENCE_PATH}')
print('\nNote: baseline paired tests use predictions from the archived implementation re-fitted in the current software environment. All archived test metrics were reproduced exactly.')


### 5C. Per-class analysis and final figures


In [ ]:
# Per-class analysis and figures
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Configuration and paths
FIGURE_DIRECTORY = os.path.join(DATA_PATH, 'thesis_final_figures')
CONFUSION_DIRECTORY = os.path.join(FIGURE_DIRECTORY, 'confusion_matrices')
os.makedirs(FIGURE_DIRECTORY, exist_ok=True)
os.makedirs(CONFUSION_DIRECTORY, exist_ok=True)
PER_CLASS_METRICS_PATH = os.path.join(DATA_PATH, 'final_per_class_metrics.csv')
PREDICTION_DISTRIBUTION_PATH = os.path.join(DATA_PATH, 'final_prediction_distributions.csv')
RAG_HYBRID_CLASS_DIFFERENCE_PATH = os.path.join(DATA_PATH, 'final_rag_hybrid_class_differences.csv')
FINAL_LABEL_KEY = os.path.join(DATA_PATH, 'final_class_label_key.csv')
MACRO_F1_FIGURE_PNG = os.path.join(FIGURE_DIRECTORY, 'final_macro_f1_comparison.png')
MACRO_F1_FIGURE_SVG = os.path.join(FIGURE_DIRECTORY, 'final_macro_f1_comparison.svg')
CLASS_F1_FIGURE_PNG = os.path.join(FIGURE_DIRECTORY, 'final_per_class_f1_heatmap.png')
CLASS_F1_FIGURE_SVG = os.path.join(FIGURE_DIRECTORY, 'final_per_class_f1_heatmap.svg')

# Reload final tables if required
if 'combined_predictions' not in globals():
    combined_predictions = pd.read_csv(COMBINED_PREDICTIONS_PATH)
if 'comparison_table' not in globals():
    comparison_table = pd.read_csv(FINAL_COMPARISON_PATH)
if 'confidence_interval_table' not in globals():
    confidence_interval_table = pd.read_csv(BOOTSTRAP_CI_PATH)
method_order = ['TF-IDF + Logistic Regression', 'Zero-shot', 'PEFT', 'RAG', 'Hybrid']
prediction_columns = {'TF-IDF + Logistic Regression': 'predicted_label_baseline', 'Zero-shot': 'predicted_label_zero_shot', 'PEFT': 'predicted_label_peft', 'RAG': 'predicted_label_rag', 'Hybrid': 'predicted_label_hybrid'}
for prediction_column in prediction_columns.values():
    if prediction_column not in combined_predictions.columns:
        raise ValueError(f'Combined prediction matrix is missing {prediction_column}. Run Cell 5F first.')
y_true = combined_predictions['clause_type'].astype(str)

# Recalculate five-method per-class metrics
per_class_records = []
for method_name in method_order:
    y_pred = combined_predictions[prediction_columns[method_name]].astype(str)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=choices, zero_division=0)
    for class_index, class_name in enumerate(choices):
        per_class_records.append({'method': method_name, 'clause_type': class_name, 'support': int(support[class_index]), 'precision': float(precision[class_index]), 'recall': float(recall[class_index]), 'f1': float(f1[class_index])})
per_class_table = pd.DataFrame(per_class_records)
per_class_table.to_csv(PER_CLASS_METRICS_PATH, index=False)

# Recalculate prediction distributions
all_prediction_labels = choices + ['Unknown']
distribution_records = []
for method_name in method_order:
    counts = combined_predictions[prediction_columns[method_name]].value_counts().reindex(all_prediction_labels, fill_value=0)
    for label_name, count in counts.items():
        distribution_records.append({'method': method_name, 'predicted_label': label_name, 'prediction_count': int(count), 'prediction_rate': float(count / len(combined_predictions))})
prediction_distribution_table = pd.DataFrame(distribution_records)
prediction_distribution_table.to_csv(PREDICTION_DISTRIBUTION_PATH, index=False)

# Preserve rag-versus-hybrid class comparison
rag_class_metrics = per_class_table.loc[per_class_table['method'].eq('RAG')].set_index('clause_type')
hybrid_class_metrics = per_class_table.loc[per_class_table['method'].eq('Hybrid')].set_index('clause_type')
rag_hybrid_class_difference = pd.DataFrame({'support': rag_class_metrics['support'], 'rag_precision': rag_class_metrics['precision'], 'hybrid_precision': hybrid_class_metrics['precision'], 'precision_difference_hybrid_minus_rag': hybrid_class_metrics['precision'] - rag_class_metrics['precision'], 'rag_recall': rag_class_metrics['recall'], 'hybrid_recall': hybrid_class_metrics['recall'], 'recall_difference_hybrid_minus_rag': hybrid_class_metrics['recall'] - rag_class_metrics['recall'], 'rag_f1': rag_class_metrics['f1'], 'hybrid_f1': hybrid_class_metrics['f1'], 'f1_difference_hybrid_minus_rag': hybrid_class_metrics['f1'] - rag_class_metrics['f1']}).reset_index()
rag_hybrid_class_difference = rag_hybrid_class_difference.sort_values('f1_difference_hybrid_minus_rag', ascending=False).reset_index(drop=True)
rag_hybrid_class_difference.to_csv(RAG_HYBRID_CLASS_DIFFERENCE_PATH, index=False)

# Save class-label key
class_label_key = pd.DataFrame({'class_id': range(1, len(choices) + 1), 'clause_type': choices})
class_label_key.to_csv(FINAL_LABEL_KEY, index=False)

# Configure publication style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')
method_colors = {'TF-IDF + Logistic Regression': '#5B6573', 'Zero-shot': '#8FBBD9', 'PEFT': '#D98F70', 'RAG': '#4C78A8', 'Hybrid': '#59A14F'}

# Regenerate macro f1 comparison with all intervals
macro_plot_data = comparison_table.set_index('method').loc[method_order].reset_index()
ci_lookup = confidence_interval_table.set_index('method')
lower_errors = []
upper_errors = []
for row in macro_plot_data.itertuples():
    lower_bound = float(ci_lookup.loc[row.method, 'macro_f1_ci_lower'])
    upper_bound = float(ci_lookup.loc[row.method, 'macro_f1_ci_upper'])
    lower_errors.append(row.macro_f1 - lower_bound)
    upper_errors.append(upper_bound - row.macro_f1)
figure, axis = plt.subplots(figsize=(10, 6.5))
bars = axis.bar(macro_plot_data['method'], macro_plot_data['macro_f1'], color=[method_colors[method] for method in macro_plot_data['method']], yerr=np.array([lower_errors, upper_errors]), capsize=5, edgecolor='black', linewidth=0.6)
axis.set_ylim(0, 0.9)
axis.set_ylabel('Macro F1')
axis.set_xlabel('')
axis.set_title('Contract clause classification performance')
axis.tick_params(axis='x', rotation=20)
for label in axis.get_xticklabels():
    label.set_horizontalalignment('right')
for bar, value in zip(bars, macro_plot_data['macro_f1']):
    axis.text(bar.get_x() + bar.get_width() / 2, value + 0.018, f'{value:.3f}', ha='center', va='bottom', fontsize=10)
axis.text(0.01, -0.26, 'Error bars show 95% paired-bootstrap intervals. Baseline row predictions come from the archived implementation re-fitted on the frozen splits; all archived test metrics were reproduced exactly.', transform=axis.transAxes, fontsize=8.5)
figure.tight_layout()
figure.savefig(MACRO_F1_FIGURE_PNG, dpi=300, bbox_inches='tight')
figure.savefig(MACRO_F1_FIGURE_SVG, bbox_inches='tight')
plt.show()

# Regenerate five-method per-class f1 heatmap
f1_matrix = per_class_table.pivot(index='method', columns='clause_type', values='f1').reindex(index=method_order, columns=choices)
figure, axis = plt.subplots(figsize=(22, 6.5))
heatmap = axis.imshow(f1_matrix.to_numpy(), cmap='YlGnBu', aspect='auto', vmin=0, vmax=1)
axis.set_xticks(np.arange(len(choices)))
axis.set_xticklabels(choices, rotation=55, ha='right', fontsize=9)
axis.set_yticks(np.arange(len(method_order)))
axis.set_yticklabels(method_order, fontsize=10)
axis.set_xlabel('Clause class')
axis.set_ylabel('Method')
axis.set_title('Per-class F1 scores across all classification methods')
for row_index in range(len(method_order)):
    for column_index in range(len(choices)):
        value = f1_matrix.iloc[row_index, column_index]
        axis.text(column_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=7, color='white' if value >= 0.65 else 'black')
colorbar = figure.colorbar(heatmap, ax=axis, fraction=0.025, pad=0.02)
colorbar.set_label('F1 score')
figure.tight_layout()
figure.savefig(CLASS_F1_FIGURE_PNG, dpi=300, bbox_inches='tight')
figure.savefig(CLASS_F1_FIGURE_SVG, bbox_inches='tight')
plt.show()


In [ ]:
# Generate normalized confusion matrices for all methods
evaluation_labels = choices + ['Unknown']
display_labels = ['Parties', 'License Grant', 'Liability Cap', 'Audit Rights', 'Anti-Assignment', 'Insurance', 'Document Name', 'Agreement Date', 'Governing Law', 'Expiration Date', 'Effective Date', 'Post-Termination', 'Revenue Sharing', 'Minimum Commitment', 'Exclusivity', 'ROFR/ROFO/ROFN', 'IP Ownership', 'Non-Transferable', 'Non-Compete', 'Termination Convenience', 'Unknown']
confusion_output_paths = []
for method_name in method_order:
    y_pred = combined_predictions[prediction_columns[method_name]].astype(str)
    full_counts = confusion_matrix(y_true, y_pred, labels=evaluation_labels)
    confusion_counts = full_counts[:len(choices), :]
    row_totals = confusion_counts.sum(axis=1, keepdims=True)
    normalized_confusion = np.divide(confusion_counts, row_totals, out=np.zeros_like(confusion_counts, dtype=float), where=row_totals != 0)
    safe_name = re.sub('[^a-z0-9]+', '_', method_name.casefold()).strip('_')
    counts_path = os.path.join(DATA_PATH, f'confusion_{safe_name}_counts.csv')
    normalized_path = os.path.join(DATA_PATH, f'confusion_{safe_name}_normalized.csv')
    figure_png_path = os.path.join(CONFUSION_DIRECTORY, f'confusion_{safe_name}_normalized.png')
    figure_svg_path = os.path.join(CONFUSION_DIRECTORY, f'confusion_{safe_name}_normalized.svg')
    pd.DataFrame(confusion_counts, index=choices, columns=evaluation_labels).to_csv(counts_path)
    pd.DataFrame(normalized_confusion, index=choices, columns=evaluation_labels).to_csv(normalized_path)
    figure, axis = plt.subplots(figsize=(18, 14))
    matrix_image = axis.imshow(normalized_confusion, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    axis.set_xticks(np.arange(len(evaluation_labels)))
    axis.set_xticklabels(display_labels, rotation=55, ha='right', fontsize=8)
    axis.set_yticks(np.arange(len(choices)))
    axis.set_yticklabels(display_labels[:-1], fontsize=8)
    axis.set_xlabel('Predicted class')
    axis.set_ylabel('True class')
    axis.set_title(f'Normalized confusion matrix: {method_name}')
    for row_index in range(len(choices)):
        for column_index in range(len(evaluation_labels)):
            value = normalized_confusion[row_index, column_index]
            if value >= 0.05:
                axis.text(column_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=6, color='white' if value >= 0.55 else 'black')
    colorbar = figure.colorbar(matrix_image, ax=axis, fraction=0.025, pad=0.02)
    colorbar.set_label('Proportion of true class')
    figure.tight_layout()
    figure.savefig(figure_png_path, dpi=300, bbox_inches='tight')
    figure.savefig(figure_svg_path, bbox_inches='tight')
    plt.close(figure)
    confusion_output_paths.extend([counts_path, normalized_path, figure_png_path, figure_svg_path])

# Final confirmation
print(f'Methods included:      {len(method_order)}')
print(f'Per-class rows:        {len(per_class_table)}')
print(f'Confusion matrices:    {len(method_order)}')
print(f'Confusion artifacts:   {len(confusion_output_paths)}')
print(f'Macro F1 figure:       {MACRO_F1_FIGURE_PNG}')
print(f'Per-class heatmap:     {CLASS_F1_FIGURE_PNG}')
print(f'Figure directory:      {FIGURE_DIRECTORY}')
display(per_class_table.loc[per_class_table['method'].eq('TF-IDF + Logistic Regression')][['clause_type', 'support', 'precision', 'recall', 'f1']].round(4))


### 5D. Build the authoritative final archive


In [ ]:
# Final archive
import glob
import hashlib

# Final output paths
FINAL_SUMMARY_JSON = os.path.join(DATA_PATH, 'final_experiment_summary.json')
FINAL_SUMMARY_TEXT = os.path.join(DATA_PATH, 'final_experiment_summary.txt')
FINAL_MANIFEST_PATH = os.path.join(DATA_PATH, 'final_experiment_file_manifest.csv')
PER_CLASS_WINNERS_PATH = os.path.join(DATA_PATH, 'final_per_class_best_method.csv')
BASELINE_METADATA_PATH = os.path.join(DATA_PATH, 'results_baseline_reproduced_metadata.json')
RAG_RETRIEVAL_METRICS_PATH = os.path.join(DATA_PATH, 'rag_v2_clean_retrieval_metrics.json')
RAG_SENSITIVITY_PATH = os.path.join(DATA_PATH, 'results_rag_v2_clean_sensitivity.csv')
HYBRID_SENSITIVITY_PATH = os.path.join(DATA_PATH, 'results_hybrid_v2_clean_sensitivity.csv')

# Reload authoritative final tables
comparison_table = pd.read_csv(FINAL_COMPARISON_PATH)
confidence_interval_table = pd.read_csv(BOOTSTRAP_CI_PATH)
mcnemar_table = pd.read_csv(MCNEMAR_RESULTS_PATH)
macro_f1_difference_table = pd.read_csv(MACRO_F1_DIFFERENCE_PATH)
per_class_table = pd.read_csv(PER_CLASS_METRICS_PATH)
rag_hybrid_class_difference = pd.read_csv(RAG_HYBRID_CLASS_DIFFERENCE_PATH)
with open(BASELINE_METADATA_PATH, 'r', encoding='utf-8') as metadata_file:
    baseline_reproduction_metadata = json.load(metadata_file)
retrieval_metrics = {}
if os.path.exists(RAG_RETRIEVAL_METRICS_PATH):
    with open(RAG_RETRIEVAL_METRICS_PATH, 'r', encoding='utf-8') as retrieval_file:
        retrieval_metrics = json.load(retrieval_file)
rag_sensitivity = pd.read_csv(RAG_SENSITIVITY_PATH)
hybrid_sensitivity = pd.read_csv(HYBRID_SENSITIVITY_PATH)

# Verify final table structure
expected_methods = ['TF-IDF + Logistic Regression', 'Zero-shot', 'PEFT', 'RAG', 'Hybrid']
if comparison_table['method'].tolist() != expected_methods:
    raise ValueError('Final comparison does not contain the expected five methods in the expected order.')
if set(confidence_interval_table['method']) != set(expected_methods):
    raise ValueError('Confidence intervals do not cover all five methods.')
if len(mcnemar_table) != 10:
    raise ValueError('Expected ten pairwise McNemar comparisons.')
if len(macro_f1_difference_table) != 10:
    raise ValueError('Expected ten paired Macro F1 comparisons.')
if len(per_class_table) != 100:
    raise ValueError('Expected 100 per-class result rows.')

# Identify the best method for each class
winner_indices = per_class_table.groupby('clause_type')['f1'].idxmax()
per_class_winners = per_class_table.loc[winner_indices, ['clause_type', 'support', 'method', 'precision', 'recall', 'f1']].rename(columns={'method': 'best_method', 'precision': 'best_precision', 'recall': 'best_recall', 'f1': 'best_f1'}).set_index('clause_type').reindex(choices).reset_index()
per_class_winners.to_csv(PER_CLASS_WINNERS_PATH, index=False)

# Extract central paired comparisons
def get_accuracy_comparison(method_a, method_b):
    result = mcnemar_table.loc[mcnemar_table['method_a'].eq(method_a) & mcnemar_table['method_b'].eq(method_b)]
    if len(result) != 1:
        raise ValueError(f'Missing or duplicated accuracy comparison: {method_a} vs {method_b}')
    return result.iloc[0]

def get_macro_comparison(method_a, method_b):
    result = macro_f1_difference_table.loc[macro_f1_difference_table['method_a'].eq(method_a) & macro_f1_difference_table['method_b'].eq(method_b)]
    if len(result) != 1:
        raise ValueError(f'Missing or duplicated Macro F1 comparison: {method_a} vs {method_b}')
    return result.iloc[0]
baseline_rag_accuracy = get_accuracy_comparison('TF-IDF + Logistic Regression', 'RAG')
baseline_hybrid_accuracy = get_accuracy_comparison('TF-IDF + Logistic Regression', 'Hybrid')
rag_hybrid_accuracy = get_accuracy_comparison('RAG', 'Hybrid')
baseline_rag_macro = get_macro_comparison('TF-IDF + Logistic Regression', 'RAG')
baseline_hybrid_macro = get_macro_comparison('TF-IDF + Logistic Regression', 'Hybrid')
rag_hybrid_macro = get_macro_comparison('RAG', 'Hybrid')

# Json-safe conversion helper
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value

# Construct authoritative final summary
final_summary = {'status': 'core_experiments_and_final_analysis_complete', 'dataset': {'training_clauses': 8257, 'validation_clauses': 833, 'test_clauses': 1850, 'classes': 20, 'split_unit': 'annotated_document_title_group', 'document_title_group_overlap': False}, 'primary_metric': 'Macro F1', 'primary_results': comparison_table.to_dict(orient='records'), 'bootstrap_confidence_intervals': confidence_interval_table.to_dict(orient='records'), 'pairwise_accuracy_tests': mcnemar_table.to_dict(orient='records'), 'paired_macro_f1_intervals': macro_f1_difference_table.to_dict(orient='records'), 'baseline_reproduction': baseline_reproduction_metadata, 'retrieval_diagnostics': retrieval_metrics, 'rag_exact_match_sensitivity': rag_sensitivity.to_dict(orient='records'), 'hybrid_exact_match_sensitivity': hybrid_sensitivity.to_dict(orient='records'), 'per_class_best_methods': per_class_winners.to_dict(orient='records'), 'central_comparisons': {'baseline_vs_rag': {'accuracy_advantage_baseline': float(-baseline_rag_accuracy['accuracy_difference_b_minus_a']), 'mcnemar_holm_adjusted_p': float(baseline_rag_accuracy['holm_adjusted_p_value']), 'macro_f1_advantage_baseline': float(-baseline_rag_macro['difference_b_minus_a']), 'macro_f1_advantage_ci_lower': float(-baseline_rag_macro['difference_ci_upper']), 'macro_f1_advantage_ci_upper': float(-baseline_rag_macro['difference_ci_lower'])}, 'baseline_vs_hybrid': {'accuracy_advantage_baseline': float(-baseline_hybrid_accuracy['accuracy_difference_b_minus_a']), 'mcnemar_holm_adjusted_p': float(baseline_hybrid_accuracy['holm_adjusted_p_value']), 'macro_f1_advantage_baseline': float(-baseline_hybrid_macro['difference_b_minus_a']), 'macro_f1_advantage_ci_lower': float(-baseline_hybrid_macro['difference_ci_upper']), 'macro_f1_advantage_ci_upper': float(-baseline_hybrid_macro['difference_ci_lower'])}, 'rag_vs_hybrid': {'accuracy_difference_hybrid_minus_rag': float(rag_hybrid_accuracy['accuracy_difference_b_minus_a']), 'mcnemar_holm_adjusted_p': float(rag_hybrid_accuracy['holm_adjusted_p_value']), 'macro_f1_difference_hybrid_minus_rag': float(rag_hybrid_macro['difference_b_minus_a']), 'macro_f1_difference_ci_lower': float(rag_hybrid_macro['difference_ci_lower']), 'macro_f1_difference_ci_upper': float(rag_hybrid_macro['difference_ci_upper'])}}, 'main_findings': ['TF-IDF with balanced logistic regression achieved the highest Accuracy, Macro F1 and Weighted F1.', 'The baseline significantly outperformed all four Mistral-based approaches on paired accuracy.', 'Paired bootstrap intervals supported higher baseline Macro F1 than RAG and Hybrid.', 'RAG and Hybrid substantially outperformed Zero-shot and PEFT.', 'RAG and Hybrid were statistically indistinguishable overall.', 'Hybrid improved performance when retrieval failed but slightly reduced performance when the correct label was retrieved.', 'Exact train-test text matches did not materially explain RAG or Hybrid Macro F1.'], 'limitations': ['The study evaluated one base language model and one legal classification dataset.', 'The PEFT configuration received limited hyperparameter exploration.', 'Generated responses using unofficial, multiple or truncated labels were treated as Unknown.', 'The baseline was re-fitted from its archived implementation to recover row-level predictions. Archived test metrics matched exactly, but minor training and validation numerical drift remained.', 'Paired Macro F1 intervals were percentile bootstrap intervals and were not adjusted for multiple pairwise comparisons.'], 'completed_at_utc': datetime.now(timezone.utc).isoformat()}
final_summary = make_json_safe(final_summary)
with open(FINAL_SUMMARY_JSON, 'w', encoding='utf-8') as summary_file:
    json.dump(final_summary, summary_file, indent=2, ensure_ascii=False, allow_nan=False)

# Save readable final summary
baseline_row = comparison_table.iloc[0]
rag_row = comparison_table.loc[comparison_table['method'].eq('RAG')].iloc[0]
hybrid_row = comparison_table.loc[comparison_table['method'].eq('Hybrid')].iloc[0]
summary_lines = ['FINAL THESIS EXPERIMENT SUMMARY', '=' * 78, '', 'DATASET', 'Training clauses:   8,257', 'Validation clauses: 833', 'Test clauses:       1,850', 'Classes:            20', 'Split unit:         Annotated document-title group', 'Title-group overlap: None', '', 'PRIMARY RESULTS', comparison_table.round(4).to_string(index=False), '', 'CENTRAL STATISTICAL FINDINGS', f"Baseline versus RAG accuracy advantage: {-baseline_rag_accuracy['accuracy_difference_b_minus_a']:.4f}; Holm-adjusted p = {baseline_rag_accuracy['holm_adjusted_p_value']:.6g}", f"Baseline versus RAG Macro F1 advantage: {-baseline_rag_macro['difference_b_minus_a']:.4f}; paired 95% CI [{-baseline_rag_macro['difference_ci_upper']:.4f}, {-baseline_rag_macro['difference_ci_lower']:.4f}]", f"Baseline versus Hybrid accuracy advantage: {-baseline_hybrid_accuracy['accuracy_difference_b_minus_a']:.4f}; Holm-adjusted p = {baseline_hybrid_accuracy['holm_adjusted_p_value']:.6g}", f"Baseline versus Hybrid Macro F1 advantage: {-baseline_hybrid_macro['difference_b_minus_a']:.4f}; paired 95% CI [{-baseline_hybrid_macro['difference_ci_upper']:.4f}, {-baseline_hybrid_macro['difference_ci_lower']:.4f}]", f"Hybrid versus RAG accuracy difference: {rag_hybrid_accuracy['accuracy_difference_b_minus_a']:+.4f}; Holm-adjusted p = {rag_hybrid_accuracy['holm_adjusted_p_value']:.6g}", f"Hybrid versus RAG Macro F1 difference: {rag_hybrid_macro['difference_b_minus_a']:+.4f}; paired 95% CI [{rag_hybrid_macro['difference_ci_lower']:.4f}, {rag_hybrid_macro['difference_ci_upper']:.4f}]", '', 'INTERPRETATION', 'The TF-IDF + Logistic Regression baseline significantly outperformed every Mistral-based approach overall.', 'RAG and Hybrid substantially outperformed Zero-shot and PEFT.', 'RAG and Hybrid were statistically indistinguishable overall.', '', 'BASELINE REPRODUCTION NOTE', 'The archived baseline implementation was re-fitted on the frozen splits to recover row-level predictions. All archived test metrics matched exactly. Minor training and validation numerical drift was observed.']
with open(FINAL_SUMMARY_TEXT, 'w', encoding='utf-8') as summary_file:
    summary_file.write('\n'.join(summary_lines))

# Build final artifact list
core_files = [os.path.join(DATA_PATH, 'train.csv'), os.path.join(DATA_PATH, 'val.csv'), os.path.join(DATA_PATH, 'test.csv'), os.path.join(DATA_PATH, 'dataset_config.json'), os.path.join(DATA_PATH, 'baseline_results.json'), os.path.join(DATA_PATH, 'results_baseline_final.csv'), os.path.join(DATA_PATH, 'results_baseline_reproduced_validation.csv'), os.path.join(DATA_PATH, 'baseline_tfidf_logistic_model.joblib'), os.path.join(DATA_PATH, 'baseline_reproduction_verification.csv'), BASELINE_METADATA_PATH, os.path.join(DATA_PATH, 'results_prompt_engineering_final.csv'), os.path.join(DATA_PATH, 'results_peft_final.csv'), os.path.join(DATA_PATH, 'results_rag_final.csv'), os.path.join(DATA_PATH, 'results_rag_peft_hybrid.csv'), COMBINED_PREDICTIONS_PATH, FINAL_COMPARISON_PATH, BOOTSTRAP_CI_PATH, BOOTSTRAP_SAMPLES_PATH, MCNEMAR_RESULTS_PATH, MACRO_F1_DIFFERENCE_PATH, PER_CLASS_METRICS_PATH, PREDICTION_DISTRIBUTION_PATH, RAG_HYBRID_CLASS_DIFFERENCE_PATH, PER_CLASS_WINNERS_PATH, RAG_RETRIEVAL_METRICS_PATH, RAG_SENSITIVITY_PATH, HYBRID_SENSITIVITY_PATH, FINAL_LABEL_KEY, FINAL_SUMMARY_JSON, FINAL_SUMMARY_TEXT, MACRO_F1_FIGURE_PNG, MACRO_F1_FIGURE_SVG, CLASS_F1_FIGURE_PNG, CLASS_F1_FIGURE_SVG]
core_files.extend(glob.glob(os.path.join(DATA_PATH, 'confusion_*_counts.csv')))
core_files.extend(glob.glob(os.path.join(DATA_PATH, 'confusion_*_normalized.csv')))
core_files.extend(glob.glob(os.path.join(CONFUSION_DIRECTORY, 'confusion_*_normalized.png')))
core_files.extend(glob.glob(os.path.join(CONFUSION_DIRECTORY, 'confusion_*_normalized.svg')))
core_files = sorted(set(core_files))

# Calculate final sha-256 checksums
def calculate_sha256(file_path):
    hash_calculator = hashlib.sha256()
    with open(file_path, 'rb') as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b''):
            hash_calculator.update(chunk)
    return hash_calculator.hexdigest()
missing_files = [file_path for file_path in core_files if not os.path.exists(file_path)]
if missing_files:
    raise FileNotFoundError('Final archive files are missing:\n' + '\n'.join(missing_files))
manifest_records = []
for file_path in core_files:
    manifest_records.append({'filename': os.path.basename(file_path), 'path': file_path, 'size_bytes': int(os.path.getsize(file_path)), 'sha256': calculate_sha256(file_path)})
manifest_table = pd.DataFrame(manifest_records)
manifest_table.to_csv(FINAL_MANIFEST_PATH, index=False)

# Final confirmation
print(f'Methods:             {len(expected_methods)}')
print(f'Test predictions:    {len(pd.read_csv(COMBINED_PREDICTIONS_PATH)):,}')
print(f'Pairwise tests:      {len(mcnemar_table)}')
print(f'Per-class rows:      {len(per_class_table)}')
print(f'Archived artifacts:  {len(manifest_table):,}')
print(f'Summary JSON:        {FINAL_SUMMARY_JSON}')
print(f'Summary text:        {FINAL_SUMMARY_TEXT}')
print(f'Checksum manifest:   {FINAL_MANIFEST_PATH}')
print('\nFinal results:')
display(comparison_table[['method', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'unknown_rate']].round(4))


## Authoritative result summary

| Method | Accuracy | Macro F1 | Weighted F1 | Unknown rate |
|---|---:|---:|---:|---:|
| TF-IDF + Logistic Regression | **0.8605** | **0.8131** | **0.8559** | 0.0000 |
| Zero-shot | 0.6227 | 0.6572 | 0.6615 | 0.1319 |
| PEFT | 0.6146 | 0.6260 | 0.6199 | 0.0011 |
| RAG | 0.8157 | 0.7762 | 0.8315 | 0.0746 |
| Hybrid | 0.8259 | 0.7764 | 0.8172 | 0.0005 |

The classical baseline achieved the highest overall scores. RAG and Hybrid substantially outperformed Zero-shot and PEFT. Hybrid's small advantage over RAG was not statistically clear: the Holm-adjusted McNemar p-value was 0.6054, and the paired Macro F1 difference interval included zero. The frozen one-epoch PEFT result therefore remains a valid negative result, while additional PEFT tuning belongs in a separately labeled sensitivity study.
